# Лабораторная работа 2: «Поправка к протоколу»

Сюжет пьесы: на допросе у следователя, проверяется факт передачи рабочей записки 3-ему лицу

Актеры: Орлов - условно следователь, Соколов - подозреваемый инженер, Волкова - проверяющая материалы


## Подготовка

In [1]:
import subprocess
import sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
    "requests>=2.32,<3", "torch>=2.6,<3", "transformers>=4.56,<5",
    "huggingface-hub>=0.34,<1", "safetensors>=0.4,<1",
    "python-dotenv>=1.0,<2",
], check=True)

CompletedProcess(args=['c:\\Users\\brata\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', 'requests>=2.32,<3', 'torch>=2.6,<3', 'transformers>=4.56,<5', 'huggingface-hub>=0.34,<1', 'safetensors>=0.4,<1', 'python-dotenv>=1.0,<2'], returncode=0)

In [2]:
import json
import hashlib
from difflib import SequenceMatcher
import os
import random
import re
import time
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path

import requests

ROOT = Path.cwd().resolve()
if not (ROOT / "LLM_Theatre.ipynb").is_file():
    raise RuntimeError("Откройте LLM_Theatre.ipynb из папки, в которой он лежит.")
# Для локальной модели используется только PyTorch.
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")

CLOUD_MODEL = os.getenv("CLOUD_MODEL", "deepseek-v4.1-flash")
LOCAL_MODEL = os.getenv("LOCAL_MODEL", "Qwen/Qwen3-1.7B")
SEED = 42
random.seed(SEED)
SCENARIO_VERSION = "protocol-v8"
EDIT_LOCAL = False
RUN_NAME = "final"
OUTPUT = ROOT / "result" / RUN_NAME
MODEL_CACHE = Path(os.getenv("THEATRE_MODEL_CACHE", str(ROOT / ".model-cache"))).expanduser().resolve()
TELEMETRY = []

## Материалы дела

Здесь собраны пять вымышленных документов. Они записываются в `result/<RUN_NAME>/case_files`, и дальше персонажи ищут сведения уже в этих файлах. Документы доступны не сразу: в первом действии объяснение и записка Орлова, во втором журнал проходной и сверка часов, в третьем - переписка.


In [3]:
# act — номер действия, с которого документ доступен персонажам.
DOCUMENTS = {
    "statement": {"file": "01_statement.txt", "act": 1, "title": "Первичное объяснение"},
    "access": {"file": "02_access.txt", "act": 2, "title": "Журнал доступа"},
    "clock": {"file": "03_clock.txt", "act": 2, "title": "Сверка часов"},
    "preliminary": {"file": "05_preliminary.txt", "act": 1, "title": "Подписанная предварительная записка"},
    "mail": {"file": "04_mail.txt", "act": 3, "title": "Переписка и состав файла"},
}

CASE_TEXTS = {
    "statement": """Дело «Север». Первичное объяснение Ильи Соколова. Все материалы вымышлены.
Соколов: «После вечернего совещания я собрался и ушёл около 20:50».
Соколов: «После этого в лабораторию не возвращался. Поехал домой».
Уточнение сотрудника: время ухода названо по памяти, запись проходной на момент объяснения не проверялась.
На столе Соколова обычно остаются рабочая папка и зарядка для телефона.
Вопрос об отправке документов в этом объяснении не задавался.
""",
    "access": """Дело «Север». Выдержка из журнала контроллера лаборатории. Все материалы вымышлены.
20:48 — выход по пропуску Ильи Соколова.
21:14 — вход по пропуску Ильи Соколова.
21:23 — выход по пропуску Ильи Соколова.
Все значения в этой выгрузке — время контроллера, а не подтверждённое точное время.
Состояние синхронизации контроллера требует отдельной сверки с техническим актом.
Срабатывание пропуска само по себе не устанавливает, кто физически им пользовался и что он делал в помещении.
""",
    "clock": """Дело «Север». Акт технической сверки часов. Все материалы вымышлены.
В интервале 20:00–22:00 контроллер лаборатории спешил на 12 минут относительно серверного времени.
Для этого интервала из времени контроллера нужно вычесть 12 минут.
Запись повторного входа 21:14 соответствует серверному времени 21:02.
Запись последнего выхода 21:23 соответствует серверному времени 21:11.
Первая запись выхода 20:48 соответствует серверному времени 20:36; объяснение «около 20:50» остаётся неточным.
У почтового сервиса время серверное, дополнительная поправка к нему не применяется.
Поправка подтверждает хронологию, но не цель возвращения и не умысел отправителя.
""",
    "preliminary": """Дело «Север». Предварительная записка Андрея Орлова. Все материалы вымышлены.
Записка составлена до получения акта сверки часов и подписана Орловым.
Рабочая версия: файл отправлен в 21:06, до повторного входа в лабораторию, отмеченного в журнале в 21:14.
Вывод записки: отправка из-за пределов лаборатории подтверждает заранее подготовленную внешнюю передачу.
Основание: выдержка времени отправки и выгрузка контроллера лаборатории.
Документов о местоположении телефона в момент отправки к записке не приложено.
Орлов подписал записку и направил её на дополнительную техническую проверку.
""",
    "mail": """Дело «Север». Выдержка из переписки, предоставленной Соколовым до беседы. Все материалы вымышлены.
Марк: «Для обсуждения консультации пришлите черновую сводку по испытаниям. Условия оплаты обсудим отдельно».
Отправка с личного телефона Соколова: 21:06 по серверному времени почтового сервиса.
Вложение: черновая сводка по испытаниям; в ней внутренние рабочие показатели.
Согласование внешней передачи этого документа в материалах отсутствует.
Подтверждения перевода денег в представленных материалах нет.
Личность Марка и его возможные связи с иностранными организациями не установлены.
Материалы не дают основания считать сам факт шпионажа доказанным.
Местоположение телефона в момент отправки в этой выдержке не зафиксировано.
""",
}

def terms(text):
    # Первые пять букв дают простое, хотя и неточное совпадение словоформ.
    return [word[:5] if len(word) > 5 else word
            for word in re.findall(r"[а-яёa-z0-9]+", text.lower())]

class CaseTools:
    def __init__(self, folder):
        self.folder = Path(folder).resolve()

    def _read(self, doc_id, act):
        if doc_id not in DOCUMENTS or DOCUMENTS[doc_id]["act"] > act:
            raise ValueError("Документ недоступен в этом действии.")
        path = (self.folder / DOCUMENTS[doc_id]["file"]).resolve()
        if not path.is_relative_to(self.folder):
            raise ValueError("Выход за каталог дела запрещён.")
        return path.read_text(encoding="utf-8").splitlines()

    def file_search(self, query, act):
        if not isinstance(query, str) or not query.strip() or len(query) > 300:
            raise ValueError("Нужен непустой запрос до 300 символов.")
        wanted = set(terms(query))
        hits = []
        for doc_id, meta in DOCUMENTS.items():
            if meta["act"] > act:
                continue
            lines = self._read(doc_id, act)
            for index, line in enumerate(lines):
                # Чем больше слов запроса нашлось в строке, тем выше она в выдаче.
                score = len(wanted.intersection(terms(line)))
                if score:
                    hits.append({"doc_id": doc_id, "file": meta["file"],
                                 "line": index + 1, "text": line, "score": score,
                                 "total_lines": len(lines)})
        return sorted(hits, key=lambda h: (-h["score"], h["file"], h["line"]))[:4]

    def read_excerpt(self, doc_id, start, end, act):
        lines = self._read(doc_id, act)
        if (type(start) is not int or type(end) is not int
                or not 1 <= start <= end <= len(lines) or end - start > 11):
            raise ValueError(f"В документе {doc_id} всего {len(lines)} строк. "
                             f"Нужно 1 <= start <= end <= {len(lines)}, не больше 12 строк за запрос.")
        return {"doc_id": doc_id, "file": DOCUMENTS[doc_id]["file"],
                "lines": [{"line": n, "text": lines[n - 1]} for n in range(start, end + 1)]}

    def timeline_check(self, recorded_time, offset_minutes):
        if (not isinstance(recorded_time, str)
                or not re.fullmatch(r"(?:[01]\d|2[0-3]):[0-5]\d", recorded_time)
                or type(offset_minutes) is not int or abs(offset_minutes) > 60):
            raise ValueError("Ожидаются время ЧЧ:ММ и поправка от -60 до 60 минут.")
        hour, minute = map(int, recorded_time.split(":"))
        # Если часы спешат на 12 минут, из их показаний вычитаем 12.
        actual = (hour * 60 + minute - offset_minutes) % (24 * 60)
        return {"recorded": recorded_time, "offset_minutes": offset_minutes,
                "actual": f"{actual // 60:02d}:{actual % 60:02d}",
                "note": "Расчёт не подтверждает верность исходной поправки; нужен документ."}

    def execute(self, name, arguments, act):
        if not isinstance(arguments, dict):
            raise ValueError("Аргументы инструмента должны быть объектом.")
        if name == "file_search" and set(arguments) == {"query"}:
            return self.file_search(arguments["query"], act)
        if name == "read_excerpt" and set(arguments) == {"doc_id", "start", "end"}:
            return self.read_excerpt(**arguments, act=act)
        # Поправку берём из акта: модель не должна подбирать её под свой вывод.
        if name == "timeline_check" and set(arguments) == {"recorded_time", "offset_minutes"} and act >= 2:
            clock_text = "\n".join(self._read("clock", act))
            offset = int(re.search(r"спешил на (\d+) минут", clock_text).group(1))
            if arguments["offset_minutes"] != offset:
                raise ValueError(f"По акту часы спешат на {offset} минут: offset_minutes должен быть +{offset}. "
                                 "Фактическое время = запись минус положительное опережение часов.")
            recorded = arguments["recorded_time"]
            if not any(line.startswith(f"{recorded} —") for line in self._read("access", act)):
                raise ValueError("Укажите время одной из записей журнала доступа.")
            return self.timeline_check(**arguments)
        raise ValueError("Неизвестный инструмент, неверные аргументы или недоступное действие.")

case_directory = OUTPUT / "case_files"
case_directory.mkdir(parents=True, exist_ok=True)
for doc_id, text in CASE_TEXTS.items():
    path = case_directory / DOCUMENTS[doc_id]["file"]
    path.write_text(text, encoding="utf-8", newline="\n")

TOOLS = CaseTools(case_directory)
TOOL_HELP = """
Доступны инструменты:
file_search(query: str) — поиск по открытым материалам дела, максимум 4 совпадения;
read_excerpt(doc_id: str, start: int, end: int) — чтение строк документа;
timeline_check(recorded_time: str, offset_minutes: int) — поправка часов, только с действия 2.
Если часы спешат, offset_minutes ПОЛОЖИТЕЛЬНЫЙ: actual = recorded_time - offset_minutes.
Значение offset_minutes обязано соответствовать акту clock; отрицательный знак для опережения неверен.
Вызов: {"tool": "file_search", "arguments": {"query": "время ухода"}}.
Ответ инструмента придёт следующим сообщением. Сам запрос не является репликой.
Готовая реплика: {"text":"...", "address_to":"sokolov", "speech_act":"question или answer или statement", "position":"Твоя текущая версия, до 35 слов"}.
question — вопрос или просьба подтвердить; answer — ответ без встречного вопроса; statement — собственная позиция.
position — личная заметка, другие персонажи её не слышат. Опирай её только на известные тебе сведения.
Не смешивай эти форматы. address_to: orlov, volkova или sokolov; только другой персонаж.
"""


## Модели

Орлов и Волкова работают через облачную модель, Соколов — через локальную Qwen. Исходные ответы Соколова сохраняются в журнале прогона.

In [ ]:
def model_snapshot_ready(snapshot):
    folder = Path(snapshot)
    def nonempty(name):
        path = folder / name
        return path.is_file() and path.stat().st_size > 0
    if not all(nonempty(name) for name in ("config.json", "tokenizer_config.json")):
        return False
    if not (nonempty("tokenizer.json") or nonempty("tokenizer.model")
            or (nonempty("vocab.json") and nonempty("merges.txt"))):
        return False
    index = folder / "model.safetensors.index.json"
    if index.is_file():
        try:
            weights = json.loads(index.read_text(encoding="utf-8"))["weight_map"]
            if not isinstance(weights, dict) or not weights:
                return False
            for name in set(weights.values()):
                if (not isinstance(name, str) or Path(name).is_absolute()
                        or ".." in Path(name).parts or not nonempty(name)):
                    return False
            return True
        except (ValueError, KeyError, TypeError, OSError):
            return False
    return nonempty("model.safetensors")


def download_model_snapshot(model_id, cache, downloader=None):
    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")
    if downloader is None:
        from huggingface_hub import snapshot_download
        downloader = snapshot_download
    options = {"cache_dir": Path(cache), "token": False,
               "allow_patterns": ["*.json", "*.safetensors", "*.txt", "*.model"]}
    revision = None
    try:
        cached = downloader(model_id, **options, local_files_only=True)
        if model_snapshot_ready(cached):
            return cached
        revision = Path(cached).name
    except (requests.RequestException, FileNotFoundError):
        pass
    for attempt in range(4):
        try:
            snapshot = downloader(model_id, **options, local_files_only=False,
                                  revision=revision, max_workers=1, etag_timeout=30, force_download=False)
            if not model_snapshot_ready(snapshot):
                revision = Path(snapshot).name
                raise requests.ConnectionError("Скачаны не все файлы модели.")
            return snapshot
        except requests.RequestException as exc:
            status = getattr(getattr(exc, "response", None), "status_code", None)
            if status is not None and status < 500 and status not in (408, 429):
                raise RuntimeError(f"Hugging Face: HTTP {status}. Проверьте имя модели и доступ.") from None
            if attempt == 3:
                raise RuntimeError("Загрузка весов прервалась после четырёх попыток. "
                                   "Кэш сохранён. Повторите запуск для докачки.") from None
            print(f"Обрыв загрузки весов. Повторная попытка {attempt + 2}/4; кэш сохранён.", flush=True)
            time.sleep(2 ** (attempt + 1))


class CloudLLM:
    def __init__(self, api_key, folder, model):
        if not api_key or not folder or api_key == "..." or folder == "...":
            raise RuntimeError("Заполните api_key и folder_id в .env.m1.")
        self.api_key, self.folder = api_key, folder
        self.model_id = model
        self.model = model if model.startswith("gpt://") else f"gpt://{folder}/{model}"

    def generate(self, system, prompt, max_tokens=600, temperature=0.55, json_mode=False, reasoning=True):
        started = time.monotonic()
        body = {"model": self.model, "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt}],
            "max_tokens": max_tokens, "temperature": temperature}
        if json_mode:
            body["response_format"] = {"type": "json_object"}
        if "deepseek" in self.model.lower():
            body["reasoning_effort"] = "low" if reasoning else "none"
            if reasoning:
                body["max_tokens"] = max(4096, max_tokens + 2048)
        for attempt in range(3):
            try:
                response = requests.post(
                    "https://ai.api.cloud.yandex.net/v1/chat/completions",
                    headers={"Authorization": f"Api-Key {self.api_key}",
                             "OpenAI-Project": self.folder}, json=body, timeout=(15, 120))
            except requests.RequestException as exc:
                if attempt < 2:
                    if isinstance(exc, requests.Timeout) and "reasoning_effort" in body:
                        body["reasoning_effort"] = "none"
                        body["max_tokens"] = max_tokens
                    time.sleep(2 ** attempt)
                    continue
                message = ("Yandex AI Studio не успел ответить: истёк тайм-аут."
                           if isinstance(exc, requests.Timeout)
                           else "Сеть недоступна: не удалось обратиться к Yandex AI Studio.")
                raise RuntimeError(message) from None
            if response.status_code == 429 or response.status_code >= 500:
                if attempt < 2:
                    time.sleep(2 ** attempt)
                    continue
            if response.status_code != 200:
                raise RuntimeError(f"Yandex AI Studio: HTTP {response.status_code}. Проверьте доступ и имя модели.")
            try:
                data = response.json()
                choice = data["choices"][0]
                text = choice["message"]["content"]
                if choice.get("finish_reason") == "length":
                    TELEMETRY.append({"backend": "cloud", "model": self.model_id,
                                      "seconds": round(time.monotonic() - started, 2),
                                      "usage": data.get("usage", {}), "finish_reason": "length",
                                      "reasoning_effort": body.get("reasoning_effort")})
                    if attempt < 2:
                        if body.get("reasoning_effort") == "low":
                            body["reasoning_effort"] = "none"
                            body["max_tokens"] = max_tokens
                        else:
                            body["max_tokens"] = min(16384, body["max_tokens"] * 2)
                        started = time.monotonic()
                        continue
                    raise RuntimeError("Облачный ответ трижды обрезан; повтор без рассуждения и увеличение лимита не помогли.")
                if not isinstance(text, str) or not text.strip():
                    raise ValueError("empty")
            except (KeyError, IndexError, ValueError, TypeError):
                raise RuntimeError("Yandex AI Studio вернул ответ без текста.") from None
            TELEMETRY.append({"backend": "cloud", "model": self.model_id,
                              "seconds": round(time.monotonic() - started, 2),
                              "usage": data.get("usage", {}), "reasoning_effort": body.get("reasoning_effort")})
            return text.strip()
        raise RuntimeError("Лимит попыток обращения к облачной модели исчерпан.")

def response_token_constraint(tokenizer, prompt_length, allowed_responses):
    # Разрешаем только продолжения заданных фраз, включая конец ответа.
    sequences = [tuple(tokenizer.encode(text, add_special_tokens=False)) + (tokenizer.eos_token_id,)
                 for text in allowed_responses]
    if not sequences or any(len(sequence) < 2 for sequence in sequences):
        raise ValueError("Нужны непустые допустимые ответы.")
    def allowed_tokens(batch_id, input_ids):
        prefix = tuple(input_ids[prompt_length:].tolist())
        return sorted({sequence[len(prefix)] for sequence in sequences
                       if len(sequence) > len(prefix) and sequence[:len(prefix)] == prefix})
    return allowed_tokens


class LocalLLM:
    def __init__(self, model_id):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.torch = torch
        self.model_id = model_id
        compatible_gpu = (torch.cuda.is_available()
                          and torch.cuda.get_device_capability()[0] >= 8
                          and torch.cuda.get_device_properties(0).total_memory >= 6 * 1024 ** 3
                          and torch.cuda.is_bf16_supported())
        self.device = "cuda" if compatible_gpu else "cpu"
        if self.device == "cpu":
            torch.set_num_threads(min(8, os.cpu_count() or 1))
        torch.manual_seed(SEED)
        if self.device == "cuda":
            torch.cuda.manual_seed_all(SEED)
        cache = MODEL_CACHE
        snapshot = download_model_snapshot(model_id, cache)
        self.tokenizer = AutoTokenizer.from_pretrained(snapshot, local_files_only=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            snapshot, dtype=torch.bfloat16 if self.device == "cuda" else torch.float32,
            attn_implementation="eager", local_files_only=True).to(self.device).eval()
        print(f"Локальная модель: {model_id}; устройство: {self.device}")

    def generate(self, system, prompt, max_tokens=180, temperature=0.65, allowed_responses=None):
        conversation = prompt if isinstance(prompt, list) else [{"role": "user", "content": prompt}]
        messages = [{"role": "system", "content": system}, *conversation]
        formatted = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        inputs = self.tokenizer(formatted, return_tensors="pt").to(self.device)
        # Ограничиваем контекст, чтобы не расходовать слишком много памяти.
        if inputs["input_ids"].shape[1] > 4096:
            raise RuntimeError("Контекст Соколова превысил 4096 токенов. Сократите историю.")
        started = time.monotonic()
        constraints = {}
        if allowed_responses is not None:
            constraints["prefix_allowed_tokens_fn"] = response_token_constraint(
                self.tokenizer, inputs["input_ids"].shape[1], allowed_responses)
        with self.torch.inference_mode():
            output = self.model.generate(
                **inputs, max_new_tokens=max_tokens, do_sample=True,
                temperature=temperature, top_p=0.8, top_k=20, repetition_penalty=1.08,
                pad_token_id=self.tokenizer.eos_token_id, **constraints)
        new_tokens = output[0, inputs["input_ids"].shape[1]:]
        text = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        if not text or len(new_tokens) >= max_tokens:
            raise RuntimeError("Локальная реплика пустая или обрезана. Проверьте промпт и лимит токенов.")
        TELEMETRY.append({"backend": "local", "model": self.model_id,
                          "device": self.device, "seconds": round(time.monotonic() - started, 2),
                          "output_tokens": len(new_tokens)})
        return text

def json_object(text):
    # Модель иногда оборачивает JSON в ```json — убираем только эту обёртку.
    value = text.strip()
    if value.startswith("```"):
        value = re.sub(r"^```(?:json)?\s*", "", value, count=1)
        value = re.sub(r"\s*```$", "", value, count=1)
    data = json.loads(value)
    if not isinstance(data, dict):
        raise ValueError("Нужен JSON-объект.")
    return data

def checked_json(llm, system, prompt, validate, max_tokens=1200, **options):
    error = ""
    for _ in range(3):
        answer = llm.generate(system, prompt + error, max_tokens=max_tokens, temperature=0.3, json_mode=True, **options)
        try:
            data = json_object(answer)
            validate(data)
            return data
        except (ValueError, TypeError, KeyError) as exc:
            error = f"\nОшибка проверки: {exc}. Верни только исправленный JSON по исходной схеме."
    raise RuntimeError(f"Модель трижды нарушила ожидаемую JSON-схему. Задача: {system[:100]}. {error.strip()}")


## Персонажи

Орлов отвечает за записку, Волкова проверяет документы, Соколов говорит о собственных действиях. Личная память Соколова доступна только его модели и проверке реплики.

In [5]:
NAMES = {"orlov": "Орлов", "volkova": "Волкова", "sokolov": "Соколов"}
PERSONAS = {
    "orlov": "Андрей Орлов, сотрудник контрразведки. Ты подписал предварительную записку и отвечаешь за неё. "
             "Говоришь коротко, прямо, иногда резко. Когда основание рушится, сам называешь, что снимаешь.",
    "volkova": "Елена Волкова, проверяющая материалы. Говоришь спокойно и точно. "
               "Начинаешь с конкретной строки или наблюдения. Отделяешь отметку пропуска от действий человека.",
    "sokolov": "Илья Соколов, инженер. Говоришь от первого лица, просто и предметно. "
               "Называешь то, что делал сам. Не превращай объяснение в оправдательную речь.",
}

LOCAL_MEMORY = {
    "return": "После совещания я ушёл, но позже вернулся за зарядкой и распечаткой.",
    "purpose": "Марк предложил обсудить консультацию. Для этого я послал сводку и думал, что черновик можно показать.",
    "permission": "Разрешения на отправку сводки Марку я не получал.",
    "reason": "В первом объяснении я скрыл возвращение: боялся, что меня спросят об отправленном рабочем файле.",
}

COMMON = """Сыграй один ход камерной пьесы. Пиши по-русски, 1–2 коротких предложения.
Ответь прямо собеседнику или сделай то, что указано для хода. Живой устный язык, без названий ролей и ремарок.
Не добавляй обстоятельств за пределами услышанного, доступных документов и личной памяти.
Обращайся к собеседникам на «вы». Уже произнесённые факты не пересказывай целиком. Запись пропуска не доказывает, где находился человек или его телефон.
Отсутствие сведений в материалах не доказывает отсутствия события."""

## План пьесы

Облачная модель распределяет эпизоды по трём действиям до появления реплик.


In [6]:
# События заданы как задачи персонажей. Слова и реакции создают модели.
SCENES = {
    1: {"focus": "Орлов предъявляет записку, а Соколов уточняет рассказ о своём уходе.",
        "docs": ("preliminary", "statement")},
    2: {"focus": "Орлову приходится решить, как отозвать вывод из подписанной записки.",
        "docs": ("clock", "access")},
    3: {"focus": "Участники выясняют цель отправки и причину умолчания.",
        "docs": ("mail",)},
}


@dataclass(frozen=True)
class Exchange:
    act: int
    speaker: str
    task: str
    target: str
    reply: str = ""
    memory: tuple[str, ...] = ()
    requires: tuple[str, ...] = ()


AGENDA = {
    "return": Exchange(
        1, "orlov", "Назови свою прежнюю версию из подписанной записки: ты считал, что файл ушёл из-за пределов лаборатории, "
        "потому что отметка входа в журнале была позже отправки. Не представляй версию как доказанный факт. "
        "Обратись к Соколову и спроси, возвращался ли он после ухода.", "sokolov",
        "Прямо скажи, что возвращался, и зачем вернулся. Не называй точное время.", ("return",)),
    "limits": Exchange(
        1, "volkova", "Объясни, что прежде чем делать вывод об отправке, нужно сверить часы проходной с серверным временем. "
        "Одна короткая реплика. Не пересказывай, зачем возвращался Соколов, и не объявляй результат сверки.", "orlov", requires=("return",)),
    "clock": Exchange(
        2, "volkova", "Сообщи, что часы спешили на 12 минут, поэтому отметке 21:14 соответствует 21:02, "
        "раньше отправки в 21:06. Обратись к Андрею Орлову и спроси, что он теперь снимает из записки. Соколова по имени не называй.", "orlov",
        "От первого лица сними прежний вывод о внешней отправке: прежний порядок отметок оказался ошибочным. "
        "Признай свою ошибку и оставь место отправки открытым. Не пересказывай порядок отметок и не повторяй цифры."),
    "purpose": Exchange(
        3, "orlov", "Назови просьбу Марка о черновой сводке и спроси Соколова, зачем он её отправил. Не упоминай телефон, время и разрешение.", "sokolov",
        "Объясни только зачем отправил: ты хотел обсудить с Марком консультацию. Начни с цели. "
        "Документ уже назван собеседником, не перечисляй содержимое и устройство.",
        ("purpose",)),
    "permission": Exchange(
        3, "volkova", "Скажи, что согласование в материалах не найдено. Спроси только, получал ли Соколов разрешение.", "sokolov",
        "Прямо ответь, получал ли ты разрешение. Скажи только то, что знаешь про своё действие.",
        ("permission",)),
    "concealment": Exchange(
        3, "orlov", "Напомни Соколову, что в первом объяснении после ухода он отрицал возвращение. "
        "Спроси, зачем он скрыл возвращение в том объяснении. Переписку в этой реплике не упоминай.", "sokolov",
        "Объясни, почему скрыл возвращение. Говори о своём страхе вопросов об отправленном файле.",
        ("reason",), ("return",)),
    "closing": Exchange(
        3, "volkova", "Коротко подведи итог: по словам Соколова, сводка ушла для обсуждения консультации без разрешения. "
        "Материалы не устанавливают место отправки или умысел. Не приписывай Соколову слова об устройстве, если он его не называл. "
        "Вопросов не задавай.", "orlov",
        requires=("purpose", "permission", "concealment")),
}

PLANNER_SYSTEM = """Ты драматург камерной пьесы «Поправка к протоколу». Выбери развитие конфликта до генерации реплик.
Верни JSON {"title":"Поправка к протоколу","acts":[{"id":1,"title":"...","goal":"...","stage":"...","topics":["return","limits"]},...]}.
Три действия в порядке номеров. В topics укажи порядок эпизодов: именно в нём они будут сыграны.
В первом действии начни с return, закончи limits. Между ними можно поставить concealment, если это обострит спор.
Во втором действии только clock: Волкова предъявляет расчёт, а Орлов пересматривает записку.
В третьем действии purpose и permission можно менять местами. Если concealment не был в первом действии,
поставь его здесь в любом порядке перед closing. Каждый ключ используется ровно один раз.
В goal кратко опиши напряжение между героями. Не пересказывай документы, цифры, хронологию и ход расчёта. В stage опиши обстановку кабинета, не добавляй новых документов. Используй только фамилии
Орлов, Волкова, Соколов. Описывай напряжение, а не обещанные реакции. Не добавляй улик, новых лиц и готовых реплик. Итоговый протокол появится после разговора."""


def validate_plan(data):
    acts = data.get("acts")
    if data.get("title") != "Поправка к протоколу" or not isinstance(acts, list) or len(acts) != len(SCENES):
        raise ValueError("Нужны название пьесы и три действия.")
    for act_id, act in zip(SCENES, acts):
        if (not isinstance(act, dict) or type(act.get("id")) is not int or act["id"] != act_id
                or any(not isinstance(act.get(key), str) or not act[key].strip() or len(act[key]) > 300
                       for key in ("title", "goal", "stage"))):
            raise ValueError("У действия нужны номер, название, цель и описание сцены.")
    if re.search(r"\d{1,2}:\d{2}|\b(?:раньше|позже|до отправки|после отправки)\b", acts[1]["goal"], re.I):
        raise ValueError("Цель второго действия должна описывать конфликт вокруг записки, а не пересказывать расчёт.")
    topics = []
    for act in acts:
        order = act.get("topics")
        if not isinstance(order, list) or not order or any(not isinstance(key, str) or key not in AGENDA for key in order):
            raise ValueError("В topics нужны ключи эпизодов из списка.")
        for key in order:
            allowed = (1, 3) if key == "concealment" else (AGENDA[key].act,)
            if act["id"] not in allowed or not set(AGENDA[key].requires) <= set(topics):
                raise ValueError(f"Эпизод {key} нарушает порядок раскрытия событий.")
            topics.append(key)
    if Counter(topics) != Counter(AGENDA.keys()) or acts[0]["topics"][-1] != "limits":
        raise ValueError("Используй каждый эпизод ровно один раз. Первое действие завершает limits.")


def make_plan(cloud):
    focus = "\n".join(f"{act}. {scene['focus']}" for act, scene in SCENES.items())
    episodes = "\n".join(f"{key}: {step.task}" for key, step in AGENDA.items())
    return checked_json(cloud, PLANNER_SYSTEM, f"{focus}\n\nЭпизоды:\n{episodes}",
                        validate_plan, 850, reasoning=False)

## Очерёдность реплик и работа с уликами

Тема из плана определяет, кто начинает эпизод. На прямой вопрос сразу отвечает адресат. Сотрудники ищут материалы по сцене и читают документы.


In [7]:
def dialogue_tail(history, limit=8):
    return "\n".join(f'{NAMES[turn["speaker"]]}: {turn["text"]}' for turn in history[-limit:]) or "Разговор ещё не начался."


def collect_evidence(cloud, tools, act, role, log):
    previous = [row for row in log if row["act"] == act and row["actor"] == role]

    def call(tool, arguments):
        result = tools.execute(tool, arguments, act)
        row = {"act": act, "actor": role, "tool": tool, "arguments": arguments, "result": result}
        log.append(row)
        previous.append(row)
        return result

    search = next((row["result"] for row in previous if row["tool"] == "file_search"), None)
    if search is None:
        def validate(data):
            query = data.get("query")
            if not isinstance(query, str) or not query.strip() or len(query) > 120:
                raise ValueError("Укажи непустой поисковый запрос до 120 символов.")

        data = checked_json(
            cloud, 'Подбери слова для поиска строк в русских материалах дела. JSON {"query":"..."}; '
                   'три-пять ключевых слов, без вопроса.',
            SCENES[act]["focus"] + " Ищи по названиям: " +
            ", ".join(DOCUMENTS[doc]["title"] for doc in SCENES[act]["docs"]),
            validate, 100, reasoning=False)
        search = call("file_search", {"query": data["query"]})

    # Выдача определяет порядок чтения. Затем дочитываем обязательные для сцены документы.
    docs = SCENES[act]["docs"]
    order = dict.fromkeys([hit["doc_id"] for hit in search if hit["doc_id"] in docs] + list(docs))
    read = {row["arguments"]["doc_id"] for row in previous if row["tool"] == "read_excerpt"}
    for doc_id in order:
        if doc_id not in read:
            call("read_excerpt", {"doc_id": doc_id, "start": 1, "end": len(tools._read(doc_id, act))})

    if act == 2 and not any(row["tool"] == "timeline_check" for row in previous):
        texts = {row["arguments"]["doc_id"]: [line["text"] for line in row["result"]["lines"]]
                 for row in previous if row["tool"] == "read_excerpt"}
        entry = next(re.match(r"(\d{2}:\d{2})", line).group(1)
                     for line in texts["access"] if "— вход по пропуску" in line)
        offset = int(re.search(r"спешил на (\d+) минут", " ".join(texts["clock"])).group(1))
        call("timeline_check", {"recorded_time": entry, "offset_minutes": offset})

    return previous

## Генерация реплик

В каждом ходе модель получает задачу, последние реплики и относящиеся к делу сведения. Подозреваемый Соколов получает выбранный фрагмент личной памяти и формулирует ответ в локальной Qwen. Проверка сверяет вопрос, ответ и источники. Отклонённый черновик не попадает в пьесу.

In [ ]:
def validate_line(text):
    if not isinstance(text, str) or not text.strip() or len(text.split()) > 45:
        raise ValueError("Нужна непустая реплика до 45 слов.")
    if "\n" in text.strip() or re.search(r"[A-Za-z]{3,}|[\u4e00-\u9fff]", text):
        raise ValueError("Одна реплика на русском языке, без служебных ссылок.")
    if any(f"{name}:" in text for name in NAMES.values()):
        raise ValueError("Не добавляй имя говорящего перед репликой.")


def reject_repeat(text, history, role):
    words = lambda value: " ".join(re.findall(r"[а-яё0-9]+", value.lower()))
    line = words(text)
    for turn in history[-8:]:
        if turn["speaker"] == role and SequenceMatcher(None, line, words(turn["text"])).ratio() >= 0.88:
            raise ValueError("Реплика повторяет предыдущую. Продолжи разговор по текущей теме.")


@dataclass
class Actor:
    role: str
    llm: object
    critic: object
    attempt_log: list = field(default_factory=list)

    def speak(self, task, speech_act, target, history, evidence, memory=()):
        local = self.role == "sokolov"
        facts = "\n".join(LOCAL_MEMORY[key] for key in memory)
        context = dialogue_tail(history, 5)
        source_text = "\n".join(
            line["text"] for event in evidence if event["tool"] == "read_excerpt" for line in event["result"]["lines"])
        calculations = [event["result"] for event in evidence if event["tool"] == "timeline_check"]
        if calculations:
            source_text += "\n" + json.dumps(calculations, ensure_ascii=False)
        system = COMMON + "\n" + PERSONAS[self.role]
        prompt = f"Задача реплики: {task}"
        if local:
            if speech_act == "answer" and history:
                prompt += "\nВопрос собеседника: " + history[-1]["text"]
            if facts:
                prompt += "\nТвоя память:\n" + facts
        else:
            prompt += "\nПоследние реплики:\n" + context
        if speech_act == "question":
            prompt += f" Адресат вопроса — {NAMES[target]}."
        if source_text:
            prompt += "\nДоступные документы и расчёты:\n" + source_text
        prompt += "\nВыведи только слова персонажа."
        correction = ""
        for attempt in range(4):
            if local and hasattr(self.llm, "torch"):
                self.llm.torch.manual_seed(SEED + len(self.attempt_log))
            options = {} if local else {"reasoning": False}
            row = {"task": task, "attempt": attempt + 1, "accepted": False}
            self.attempt_log.append(row)
            try:
                raw = self.llm.generate(system, prompt + correction, max_tokens=85 if local else 150,
                                        temperature=0.35 + attempt * 0.1, **options)
                row["raw_text"] = raw
                text = raw.strip().strip("«»").strip()
                validate_line(text)
                if text.count("?") != (1 if speech_act == "question" else 0):
                    raise ValueError("Задай один прямой вопрос." if speech_act == "question" else "Ответь без встречного вопроса.")
                reject_repeat(text, history, self.role)
                if speech_act == "question" and target == "orlov" and re.search(r"\b(?:илья|соколов)\b", text, re.I):
                    raise ValueError("Адресуй вопрос Андрею Орлову, автору записки.")
                if self.role == "orlov" and not history:
                    version = text.lower()
                    if (not re.search(r"из-за пределов|за пределами|внешн|\bизвне\b|\bснаружи\b", version)
                            or re.search(r"не (?:утвержда|делал вывода|указа|говорил|устанавлива)", version)):
                        raise ValueError("Назови исходную версию записки об отправке извне и её основание.")
                if self.role == "orlov" and speech_act == "answer" and calculations:
                    if re.search(r"\d|\b(?:до|после|раньше|позже|прежде)\b", text, re.I):
                        raise ValueError("Сними вывод о внешней отправке, не пересказывая порядок событий.")
                if self.role == "orlov" and speech_act == "question" and "просьбу Марка" in task:
                    if re.search(r"телефон|\d|согласован|разрешен|\bназовите\b", text, re.I):
                        raise ValueError("Назови просьбу Марка и спроси только о цели отправки, без устройства, времени и разрешения.")
                if "purpose" in memory and not re.search(r"консультац|посоветова|обсудить.{0,30}работ", text, re.I):
                    raise ValueError("Назови цель отправки — обсуждение консультации с Марком.")
                self.check_line(task, speech_act, context, text, facts, source_text)
                row["accepted"] = True
                return {"text": text, "raw_text": raw, "address_to": target, "speech_act": speech_act,
                        "backend": "local" if local else "cloud", "generation_attempts": attempt + 1}
            except ValueError as exc:
                row["reason"] = str(exc)
                correction = "\nПредыдущий вариант: " + row.get("raw_text", "") + "\nЗамечание: " + str(exc)
            except RuntimeError as exc:
                if not local or "пустая или обрезана" not in str(exc):
                    raise
                row["reason"] = str(exc)
                correction = "\nУмести ответ в одно короткое предложение."
        raise RuntimeError(f"{NAMES[self.role]}: {self.attempt_log[-1]['reason']}")

    def check_line(self, task, speech_act, context, text, facts, sources):
        def validate(data):
            if type(data.get("valid")) is not bool or not isinstance(data.get("reason"), str):
                raise ValueError("Нужны valid и reason.")

        check = checked_json(
            self.critic, 'Проверь реплику пьесы. JSON {"valid":true,"reason":""} или '
                         '{"valid":false,"reason":"одно краткое замечание"}. '
                         'Реплика должна выполнить задачу, ответить на предыдущий вопрос, если это ответ, '
                         'и не добавлять новых фактов. Вопрос должен касаться ровно указанного предмета. '
                         'Проверь каждое обязательное сведение. Различай отрицание, предположение и утверждение. Не путай первичное объяснение с перепиской. '
                         'Если персонаж знает ответ, отклони уклонение. Не требуй дословных формулировок. '
                         'Не считай исправление хронологии доказательством местоположения или умысла.',
            json.dumps({"role": NAMES[self.role], "task": task, "type": speech_act, "recent": context,
                        "memory": facts, "sources": sources, "line": text}, ensure_ascii=False),
            validate, 200, reasoning=False)
        if not check["valid"]:
            raise ValueError(check["reason"] or "Реплика не выполняет задачу.")

## Итоговый протокол

Код собирает протокол из принятых реплик, прочитанных строк и расчёта поправки. Показания сопровождаются ссылкой на конкретную реплику. Каждый персонаж оценивает один пункт, после согласие и возражения персонажей сверяются с источниками. Если модель исчерпала попытки проверки, в протоколе ставиться отметка.

In [ ]:
def stage_direction(cloud, stage, history, opening):
    text = cloud.generate(
        "Ты рассказчик. В сцене только Орлов, Волкова и Соколов. Пиши по-русски. Одна ремарка до 35 слов, только наблюдаемые действия и паузы. "
        "В начале покажи, как участники готовятся говорить, в конце покажи реакцию на последнее сказанное. "
        "Выбери одно-два действия. Не повторяй ремарку предыдущей сцены, не добавляй улик, мыслей или реплик.",
        f'Сцена: {stage}\nНачало: {opening}\n{dialogue_tail(history, 4)}',
        max_tokens=180, temperature=0.4, reasoning=False)
    return text.strip().strip("*")

def source_catalog(history, tool_log):
    # Ссылки протокола могут вести только на реплики, найденные строки и расчёты.
    catalog = {f"turn:{i}": {"kind": "testimony", "speaker": NAMES[t["speaker"]], "text": t["text"]}
               for i, t in enumerate(history, 1)}
    for i, event in enumerate(tool_log, 1):
        result = event["result"]
        if event["tool"] == "file_search":
            rows = [(h["doc_id"], h["line"], h["text"]) for h in result]
        elif event["tool"] == "read_excerpt":
            rows = [(result["doc_id"], h["line"], h["text"]) for h in result["lines"]]
        else:
            rows = []
            if event["tool"] == "timeline_check":
                catalog[f"calc:{i}"] = {"kind": "calculation", "text": json.dumps(result, ensure_ascii=False)}
        for doc_id, line, text in rows:
            catalog[f"{doc_id}:{line}"] = {"kind": "document", "text": text}
    return catalog

SECTIONS = {"established": "Факты и показания", "corrected": "Исправлено", "unresolved": "Требует проверки"}

def validate_protocol(data, catalog):
    if set(data) != set(SECTIONS):
        raise ValueError("Нужны ровно три раздела протокола.")
    # Здесь проверяются ссылки и формат. Смысл отдельно оценивает модель.
    correction_refs = []
    for section in SECTIONS:
        rows = data[section]
        if not isinstance(rows, list) or not 1 <= len(rows) <= 5:
            raise ValueError("В каждом разделе от 1 до 5 записей.")
        for row in rows:
            if not isinstance(row, dict) or set(row) != {"statement", "sources"}:
                raise ValueError("Нужны statement и sources.")
            if not isinstance(row["statement"], str) or not row["statement"].strip() or len(row["statement"]) > 500:
                raise ValueError("Нужна краткая формулировка.")
            if re.search(r"\b(?:turn|calc|statement|access|clock|mail|preliminary):\d+\b", row["statement"]):
                raise ValueError("Пункт должен быть полным предложением. Ссылки укажи только в sources.")
            refs = row["sources"]
            if (not isinstance(refs, list) or not refs or len(refs) > 6
                    or any(not isinstance(ref, str) or ref not in catalog for ref in refs)):
                raise ValueError("Ссылайся только на существующие источники из каталога.")
            if section == "corrected":
                correction_refs.extend(refs)
    if not all(any(ref.startswith(prefix) for ref in correction_refs) for prefix in ("preliminary:", "clock:")):
        raise ValueError("Исправление должно ссылаться на исходную записку и акт сверки.")


def validate_review_text(review):
    validate_line(review.get("text"))
    if len(review["text"].split()) < 4:
        raise ValueError("Объясни согласие или возражение: назови конкретную часть пункта и причину.")
    if re.search(r"формулировка согласна|пункт согласен", review["text"], re.I):
        raise ValueError("Нужен понятный отзыв по существу пункта, а не общая фраза.")


def check_review_meaning(critic, role, review, statement, sources):
    if "mail:5" in sources and re.search(
            r"не\s+(?:запрашивал\w*|просил\w*|выдавал\w*|выдал\w*|был[оа]?\s+выдан\w*)", review["text"], re.I):
        raise ValueError("Из источников известно, что Соколов не получил разрешение. Обращался ли он за ним и выдавалось ли оно, не установлено.")
    def validate(data):
        if type(data.get("valid")) is not bool or not isinstance(data.get("reason"), str):
            raise ValueError("Нужны valid и reason.")
    return checked_json(critic,
        'Проверь отзыв на единственный пункт. JSON {"valid":true или false,"reason":"..."}. '
        'accept означает согласие с формулировкой этого пункта, object — конкретное возражение к ней. '
        'Отклони отзыв, который приписывает пункту отсутствующие в нём слова, смешивает его с другими пунктами '
        'или противоречит собственному статусу, личной памяти или приложенным источникам. '
        'Проверь также каждую предлагаемую замену. Отсутствие записи означает только отсутствие записи. '
        'Фраза «он ответил» вводит показание, а не независимое подтверждение факта. '
        'Фраза «не получил разрешение» не доказывает, что он не запрашивал его или что его не выдавали. '
        'Если отзыв ограничивается новой гипотезой, в тексте она должна оставаться гипотезой. '
        'Нужна понятная фраза от лица участника с конкретной причиной. «Формулировка согласна» не является отзывом. '
        'Слова «согласен» без объяснения недостаточны. Не требуй литературного стиля.',
        json.dumps({"statement": statement, "sources": sources, "review": review, "personal_memory": LOCAL_MEMORY if role == "sokolov" else PERSONAS[role]},
                   ensure_ascii=False), validate, 250, reasoning=False)

def review_protocol(actor, draft, catalog, history):
    items = {
        f"{section}:{i}": row["statement"]
        for section, rows in draft.items() for i, row in enumerate(rows, 1)
    }
    points = {
        f"{section}:{i}": {"statement": row["statement"],
                            "evidence": [catalog[ref]["text"] for ref in row["sources"]]}
        for section, rows in draft.items() for i, row in enumerate(rows, 1)
    }
    prompt = ("Выбери один пункт и оцени его по приложенным к нему источникам.\n"
              + json.dumps(points, ensure_ascii=False))
    system = (PERSONAS[actor.role] + '\nЭто отдельная проверка формулировок, а не ход беседы. '
              'Объясни согласие или возражение к формулировке пункта. В quote скопируй короткий фрагмент пункта, который оцениваешь. '
              'Любая предложенная поправка должна следовать из приложенных источников. Если вопрос открыт, так и скажи. '
              'JSON {"status":"accept" или "object","item":"ключ пункта","quote":"точный фрагмент пункта","text":"до 45 слов"}.')
    def validate(data):
        if data.get("status") not in ("accept", "object") or data.get("item") not in items:
            raise ValueError("Нужны status и существующий item.")
        validate_review_text(data)
        if data["status"] == "object" and data["item"].startswith("corrected:") and re.search(
                r"порядок (?:отметок|событий)", data["text"], re.I) and not re.search(r"\d{2}:\d{2}", data["text"]):
            raise ValueError("Если оспариваешь порядок отметок, назови конкретное время и объясни, почему оно не сходится.")
        if actor.role != "sokolov":
            quote = data.get("quote")
            if not isinstance(quote, str) or len(quote.split()) < 2 or quote not in items[data["item"]]:
                raise ValueError("В quote дословно скопируй несколько слов из выбранного пункта.")
        section, number = data["item"].split(":")
        refs = draft[section][int(number) - 1]["sources"]
        evidence = {ref: catalog[ref] for ref in refs}
        check = check_review_meaning(actor.critic or actor.llm, actor.role, data, items[data["item"]], evidence)
        if not check["valid"]:
            raise ValueError(check["reason"] or "Отзыв не следует из выбранного пункта и источников.")
    if actor.role == "sokolov":
        corrected = draft["corrected"]
        index = next((i for i, row in enumerate(corrected)
                      if any(ref.startswith("clock:") for ref in row["sources"])), 0)
        focus = f"corrected:{index + 1}"
        sources = "\n".join(catalog[ref]["text"] for ref in corrected[index]["sources"])
        system = ('Ты Соколов, участник обсуждения. Сейчас ты читаешь один пункт протокола. '
                  'Это отдельная проверка формулировок, а не ответ на допросе. '
                  'Верни JSON {"status":"accept" или "object","text":"до 35 слов"}. '
                  'accept: назови, с какой частью пункта согласен и почему. object: назови конкретную ошибку. '
                  'Ответ — полное предложение от первого лица, не меньше четырёх слов. '
                  'Незнание точного времени само по себе не является возражением. '
                  'Перескажи одну конкретную мысль пункта как причину своего согласия или возражения. '
                  'Если место телефона не установлено, не пиши, что оно указано или подтверждено. '
                  'Оцени только написанное; не приписывай пункту противоположный смысл. '
                  'Служебные ссылки в тексте отзыва не произноси.')
        prompt = ("Этот пункт протокола зачитан вслух. Согласен с его формулировкой или возражаешь?\n"
                  + items[focus] + "\nИсточники этого пункта:\n" + sources)
        correction = ""
        for attempt in range(4):
            if hasattr(actor.llm, "torch"):
                actor.llm.torch.manual_seed(SEED + len(actor.attempt_log))
            row = {"kind": "protocol_review", "attempt": attempt + 1, "accepted": False}
            actor.attempt_log.append(row)
            try:
                raw = actor.llm.generate(system, prompt + correction, max_tokens=125,
                                         temperature=0.35 + attempt * 0.1)
                row["raw_text"] = raw
                data = json_object(raw)
            except RuntimeError as exc:
                if "пустая или обрезана" not in str(exc):
                    raise
                row["reason"] = str(exc)
                correction = "\nВерни один короткий ответ в JSON."
                continue
            except ValueError as exc:
                row["reason"] = str(exc)
                correction = "\nВерни только JSON с полями status и text."
                continue
            try:
                data["item"] = focus
                validate(data)
                row["accepted"] = True
                return {"speaker": actor.role, **data, "raw_text": raw, "backend": "local"}
            except (ValueError, TypeError, KeyError) as exc:
                row["reason"] = str(exc)
                correction = f'\nИсправь отзыв: {exc}. Оцени только приведённую формулировку, без новых событий.'
        return {"speaker": actor.role, "status": "abstain", "item": focus,
                "text": "Я воздержусь от оценки этого пункта.", "backend": "scripted"}
    try:
        data = checked_json(actor.llm, system, prompt, validate, 400, reasoning=False)
    except RuntimeError as exc:
        if "нарушила ожидаемую JSON-схему" not in str(exc):
            raise
        return {"speaker": actor.role, "status": "abstain", "item": "corrected:1",
                "text": "Отзыв не прошёл проверку по источникам.", "backend": "validation"}
    return {"speaker": actor.role, **data, "backend": "cloud"}



def build_protocol(history, tool_log):
    # Показания берём из принятых реплик, а ссылки — из журнала и документов.
    turns = {(turn["topic"], turn["speech_act"]): (number, turn)
             for number, turn in enumerate(history, 1)}
    calculations = [(number, event["result"]) for number, event in enumerate(tool_log, 1)
                    if event["tool"] == "timeline_check"]
    calculation, time = calculations[0]

    def testimony(topic):
        number, turn = turns[(topic, "answer")]
        return f"turn:{number}", turn["text"].rstrip(".")

    returned, return_text = testimony("return")
    purpose, purpose_text = testimony("purpose")
    permission, permission_text = testimony("permission")
    concealment, concealment_text = testimony("concealment")
    corrected, _ = testimony("clock")
    return {
        "established": [
            {"statement": f"В первом объяснении Соколов отрицал возвращение. В разговоре он уточнил: «{return_text}».",
             "sources": ["statement:3", returned]},
            {"statement": f"О цели отправки Соколов сказал: «{purpose_text}».",
             "sources": ["mail:2", purpose]},
            {"statement": f"На вопрос, получал ли он разрешение, Соколов ответил: «{permission_text}». "
                          "В переписке согласование не зафиксировано.",
             "sources": ["mail:5", f"turn:{turns[('permission', 'question')][0]}", permission]},
            {"statement": f"Причину умолчания о возвращении Соколов объяснил так: «{concealment_text}».",
             "sources": [concealment]},
        ],
        "corrected": [
            {"statement": f"Орлов снял вывод о внешней отправке. В записке отметка входа {time['recorded']} шла после отправки "
                          f"в 21:06. По акту ей соответствует {time['actual']}, то есть отметка входа теперь раньше отправки.",
             "sources": ["preliminary:3", "preliminary:4", "clock:4", f"calc:{calculation}", "mail:3", corrected]},
        ],
        "unresolved": [
            {"statement": "Сверка часов не устанавливает местонахождение телефона в момент отправки.",
             "sources": ["access:7", "mail:9"]},
            {"statement": "Личность Марка и его связи не установлены. Материалы не доказывают умысел на шпионаж.",
             "sources": ["mail:7", "mail:8"]},
        ],
    }


def finalize_protocol(cloud, actors, history, tool_log, state, save):
    catalog = source_catalog(history, tool_log)
    state["scope"] = "reviewed_items"
    state["sources"] = catalog
    rounds = state.setdefault("rounds", [])
    if not rounds:
        draft = build_protocol(history, tool_log)
        validate_protocol(draft, catalog)
        rounds.append({"draft": draft, "reviews": []})
        save()
    current = rounds[0]
    for role, actor in actors.items():
        if not any(review["speaker"] == role for review in current["reviews"]):
            current["reviews"].append(review_protocol(actor, current["draft"], catalog, history))
            save()
    state["status"] = "agreed" if all(review["status"] == "accept" for review in current["reviews"]) else "disputed"
    save()
    return state


def run_signature():
    # Сохраняем версию кода, документов и настроек в результатах прогона.
    notebook = json.loads((ROOT / "LLM_Theatre.ipynb").read_text(encoding="utf-8"))
    code = ["".join(c["source"]) for c in notebook["cells"]
            if c["cell_type"] == "code"
            and not {"setup", "live", "export"}.intersection(c.get("metadata", {}).get("tags", []))]
    documents = {k: (TOOLS.folder / v["file"]).read_text(encoding="utf-8") for k, v in DOCUMENTS.items()}
    payload = {"version": SCENARIO_VERSION, "code": code, "documents": documents,
               "cloud": CLOUD_MODEL, "local": LOCAL_MODEL, "edit_local": EDIT_LOCAL, "seed": SEED}
    return hashlib.sha256(json.dumps(payload, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # Сначала записываем временный файл, затем заменяем предыдущий целиком.
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)

def measure_run(history, tool_log, decisions, scene_checks, protocol, telemetry):
    repeats = []
    for i, turn in enumerate(history):
        try:
            reject_repeat(turn["text"], history[:i], turn["speaker"])
        except ValueError:
            repeats.append(i + 1)
    return {"turns": len(history), "roles": dict(Counter(t["speaker"] for t in history)),
            "backends": dict(Counter(t["backend"] for t in history)),
            "tools": dict(Counter(e["tool"] for e in tool_log)),
            "speaker_selection": dict(Counter(d["source"] for d in decisions)),
            "model_calls": dict(Counter(t["backend"] for t in telemetry)),
            "model_seconds": round(sum(t["seconds"] for t in telemetry), 2),
            "near_repeat_turns": repeats,
            "questions_between_officers": sum(t.get("speech_act") == "question" and t["speaker"] != "sokolov"
                and t.get("address_to") != "sokolov" for t in history),
            "sokolov_questions": sum("?" in t["text"] for t in history if t["speaker"] == "sokolov"),
            "scene_endings": {k: v["ending"] for k, v in scene_checks.items()},
            "protocol_status": protocol["status"], "protocol_rounds": len(protocol["rounds"])}

def export_play(plan, history, directions, decisions, tool_log, output, telemetry,
                protocol, scene_checks, signature):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    metrics = measure_run(history, tool_log, decisions, scene_checks, protocol, telemetry)
    run = {"scenario_version": SCENARIO_VERSION, "signature": signature,
           "title": plan["title"], "created_at": datetime.now(timezone.utc).isoformat(),
           "seed": SEED, "models": {"cloud": CLOUD_MODEL, "local": LOCAL_MODEL}, "edit_local": EDIT_LOCAL,
           "plan": plan, "turns": history, "directions": directions,
           "decisions": decisions, "tool_calls": tool_log, "telemetry": list(telemetry),
           "scene_checks": scene_checks, "protocol": protocol, "metrics": metrics}
    write_json(output / "run.json", run)
    return run


def play_markdown(run):
    plan, history = run["plan"], run["turns"]
    directions, protocol = run["directions"], run["protocol"]
    lines = [f'# {plan["title"]}', "", "Вымышленная сцена. Результат генерации моделей.", ""]
    for act in plan["acts"]:
        key = str(act["id"])
        lines.extend([f'## Действие {act["id"]}. {act["title"]}', "", f'*{directions[key]["opening"]}*', ""])
        for turn in history:
            if turn["act"] == act["id"]:
                lines.extend([f'**{NAMES[turn["speaker"]]}:** {turn["text"]}', ""])
        lines.extend([f'*{directions[key]["closing"]}*', ""])
    lines.extend(["## Проект итогового протокола", "", "*Формулировки и источники зачитаны всем участникам.*", ""])
    final = protocol["rounds"][-1]
    for section, title in SECTIONS.items():
        lines.extend([f"### {title}", ""])
        for row in final["draft"][section]:
            lines.append(f'- {row["statement"]}')
            for ref in row["sources"]:
                lines.append(f'  - `{ref}`: {protocol["sources"][ref]["text"]}')
        lines.append("")
    lines.extend(["### Отзывы участников", ""])
    for review in final["reviews"]:
        status = {"accept": "согласна" if review["speaker"] == "volkova" else "согласен",
                  "object": "возражает", "abstain": "воздержалась" if review["speaker"] == "volkova" else "воздержался"}[review["status"]]
        lines.extend([f'**{NAMES[review["speaker"]]} ({status}):** {review["text"]}', ""])
    lines.append("Статус: " + ("по рассмотренным пунктам возражений нет." if protocol["status"] == "agreed" else "согласие по рассмотренным пунктам не достигнуто."))
    return "\n".join(lines) + "\n"



def validate_resume(saved, signature):
    if saved and (saved.get("scenario_version") != SCENARIO_VERSION or saved.get("signature") != signature):
        raise RuntimeError("Сохранённое состояние не соответствует текущей версии пьесы.")


def run_theatre(cloud, local, plan, output=OUTPUT, tools=TOOLS, resume=None):
    validate_plan(plan)
    signature = run_signature()
    validate_resume(resume, signature)
    if resume and resume["plan"] != plan:
        raise ValueError("Для продолжения нужен сохранённый план.")
    state = resume if resume is not None else {
        "scenario_version": SCENARIO_VERSION, "signature": signature, "plan": plan,
        "turns": [], "decisions": [], "tool_calls": [], "directions": {}, "scene_checks": {},
        "protocol": {}, "completed_topics": [], "active_topic": None, "actor_attempts": {}, "telemetry": [],
    }
    history, log, decisions = state["turns"], state["tool_calls"], state["decisions"]
    directions, scene_checks = state["directions"], state["scene_checks"]
    prior_telemetry = state["telemetry"]
    telemetry_start = len(TELEMETRY) if resume else 0
    actors = {role: Actor(role, local if role == "sokolov" else cloud, cloud,
                          state["actor_attempts"].setdefault(role, [])) for role in NAMES}

    def checkpoint(complete=False):
        state["complete"] = complete
        state["telemetry"] = prior_telemetry + TELEMETRY[telemetry_start:]
        write_json(Path(output) / "checkpoint.json", state)

    def speak(act_id, role, target, task, speech_act, topic, source, evidence, memory=()):
        utterance = actors[role].speak(task, speech_act, target, history, evidence, memory)
        history.append({"act": act_id, "speaker": role, "topic": topic, **utterance})
        decisions.append({"turn": len(history), "speaker": role, "source": source, "topic": topic})
        print(f'{NAMES[role]}: {utterance["text"]}', flush=True)
        checkpoint()

    try:
        for act in plan["acts"]:
            act_id = act["id"]
            key = str(act_id)
            direction = directions.setdefault(key, {})
            if "closing" in direction:
                continue
            if "opening" not in direction:
                direction["opening"] = stage_direction(cloud, act["stage"], history, True)
                checkpoint()
            print(f'\nДействие {act_id}. {act["title"]}\n*{direction["opening"]}*', flush=True)
            completed = set(state["completed_topics"])
            for topic in act["topics"]:
                if topic in completed:
                    continue
                state["active_topic"] = {"topic": topic, "source": "planner"}
                checkpoint()
                step = AGENDA[topic]
                previous = [turn for turn in history if turn["topic"] == topic]
                if not previous:
                    evidence = collect_evidence(cloud, tools, act_id, step.speaker, log)
                    if topic == "purpose":
                        # Для вопроса о цели нужны просьба Марка и описание вложения.
                        evidence = [{"tool": "read_excerpt", "result": {"lines": [
                            line for event in evidence if event["tool"] == "read_excerpt"
                            and event["arguments"]["doc_id"] == "mail"
                            for line in event["result"]["lines"] if line["line"] in (2, 4)
                        ]}}]
                    speak(act_id, step.speaker, step.target, step.task, "question" if step.reply else "statement",
                          topic, "planner", evidence, step.memory if not step.reply else ())
                if step.reply and len(previous) < 2:
                    evidence = [row for row in log if row["act"] == act_id] if step.target != "sokolov" else []
                    speak(act_id, step.target, step.speaker, step.reply, "answer", topic, "rule", evidence, step.memory)
                completed.add(topic)
                state["completed_topics"].append(topic)
                state["active_topic"] = None
                checkpoint()
            direction["closing"] = stage_direction(cloud, act["stage"], history, False)
            scene_checks[key] = {"ending": "goal", "checks": [
                {"complete": True, "after_turn": len(history), "topics": act["topics"]}]}
            checkpoint()
        protocol = finalize_protocol(cloud, actors, history, log, state["protocol"], checkpoint)
        result = export_play(plan, history, directions, decisions, log, output,
                             prior_telemetry + TELEMETRY[telemetry_start:], protocol, scene_checks, signature)
        checkpoint(complete=True)
        return result
    except (Exception, KeyboardInterrupt):
        checkpoint()
        raise

## Запуск пьесы

Результаты пишутся в `result/final/`. Каждый запуск этой ячейки удаляет предыдущие результаты в папке прогона и вычисляет пьесу заново. Менять `RUN_NAME` не нужно. Кэш весов моделей сохраняется.

In [30]:
import shutil
from dotenv import dotenv_values

credentials = dotenv_values(ROOT / ".env.m1")
FOLDER = credentials.get("folder_id", "")
API_KEY = credentials.get("api_key", "")
cloud = CloudLLM(API_KEY, FOLDER, CLOUD_MODEL)

# Очищаем только папку прогона внутри result, сохраняя кэш моделей.
result_directory = (ROOT / "result").resolve()
output_directory = OUTPUT.resolve()
if (not result_directory.is_relative_to(ROOT.resolve())
        or output_directory == result_directory
        or not output_directory.is_relative_to(result_directory)):
    raise ValueError("Папка прогона должна находиться внутри result.")
if MODEL_CACHE.resolve().is_relative_to(output_directory):
    raise ValueError("Кэш моделей должен находиться вне папки прогона.")
if output_directory.exists():
    shutil.rmtree(output_directory)
run = None
TELEMETRY.clear()
random.seed(SEED)

# Материалы дела нужны инструментам и после очистки результатов.
case_directory = output_directory / "case_files"
case_directory.mkdir(parents=True, exist_ok=True)
for doc_id, text in CASE_TEXTS.items():
    (case_directory / DOCUMENTS[doc_id]["file"]).write_text(
        text, encoding="utf-8", newline="\n")
TOOLS = CaseTools(case_directory)
print(f"Новый прогон: {output_directory.relative_to(ROOT)}.")

local = LocalLLM(LOCAL_MODEL)
plan = make_plan(cloud)
for act in plan["acts"]:
    titles = {"return": "возвращение", "limits": "необходимость сверки", "clock": "пересмотр записки",
              "purpose": "цель отправки", "permission": "разрешение", "concealment": "причина умолчания",
              "closing": "итог разговора"}
    print(f'{act["id"]}. {act["title"]} — {", ".join(titles[topic] for topic in act["topics"])}')

run = run_theatre(cloud, local, plan, output=output_directory, tools=TOOLS)


c:\Users\brata\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:17<00:00,  8.69s/it]


Локальная модель: Qwen/Qwen3-1.7B; устройство: cpu
1. Сомнение в записке — возвращение, причина умолчания, необходимость сверки
2. Пересмотр времени — пересмотр записки
3. Цель и разрешение — цель отправки, разрешение, итог разговора

Действие 1. Сомнение в записке
*Орлов стоит у окна спиной к столу. Волкова садится на ближний стул, кладёт папку на колени. Соколов останавливается у торца стола, ладонью касается спинки стула.*
Орлов: Соколов, в записке я исходил из того, что файл ушёл до повторного входа в лабораторию — значит, извне. Скажите прямо: после ухода вы возвращались?


RuntimeError: Соколов: Ответь без встречного вопроса.

## Результат прогона

Здесь выводятся статистика, текст пьесы и итоговый протокол. При повторном выполнении этой ячейки вывод обновляется по текущему прогону.


In [11]:
from IPython.display import Markdown, display

print(json.dumps(run["metrics"], ensure_ascii=False, indent=2))
assert set(t["speaker"] for t in run["turns"]) == set(NAMES)
for turn in run["turns"]:
    if turn["speaker"] == "sokolov":
        assert turn["backend"] in ("local", "local+cloud_edit", "local_constrained") and turn["raw_text"]
assert all(e["actor"] != "sokolov" for e in run["tool_calls"])
for question, answer in zip(run["turns"], run["turns"][1:]):
    if question["speech_act"] == "question":
        assert answer["speaker"] == question["address_to"] and answer["topic"] == question["topic"]
assert set(r["speaker"] for r in run["protocol"]["rounds"][-1]["reviews"]) == set(NAMES)
validate_protocol(run["protocol"]["rounds"][-1]["draft"], run["protocol"]["sources"])

unfinished = [key for key, scene in run["scene_checks"].items()
              if not scene["checks"] or not scene["checks"][-1]["complete"]]
if unfinished:
    print("Нужна проверка перед сдачей: не достигнуты цели действий " + ", ".join(unfinished))
for review in run["protocol"]["rounds"][-1]["reviews"]:
    if review["status"] != "abstain":
        validate_review_text(review)

# Ремарки привязаны к событиям пьесы, поэтому сохраняются при смене порядка эпизодов.
DELIVERY = {
    ("concealment", "sokolov", "answer"): ("тише, после паузы", "После паузы Соколов говорит тише."),
    ("clock", "volkova", "question"): ("указывая на журнал, отчётливо", "Волкова указывает на журнал и говорит отчётливо."),
    ("clock", "orlov", "answer"): ("сдержанно, после паузы", "Орлов выдерживает паузу и отвечает сдержанно."),
}


def delivery_cue(turn):
    return DELIVERY.get((turn.get("topic"), turn["speaker"], turn.get("speech_act")))


# Пометка идёт перед словами персонажа и сохраняется в текстовом выводе.
def play_with_delivery(run):
    play = play_markdown(run)
    for turn in run["turns"]:
        cue = delivery_cue(turn)
        if cue:
            line = f'**{NAMES[turn["speaker"]]}:** {turn["text"]}'
            play = play.replace(line, f'**{NAMES[turn["speaker"]]}** *({cue[0]}):* {turn["text"]}', 1)
    return play


display(Markdown(play_with_delivery(run)))

{
  "turns": 12,
  "roles": {
    "orlov": 4,
    "sokolov": 4,
    "volkova": 4
  },
  "backends": {
    "cloud": 8,
    "local": 4
  },
  "tools": {
    "file_search": 5,
    "read_excerpt": 8,
    "timeline_check": 1
  },
  "speaker_selection": {
    "planner": 7,
    "rule": 5
  },
  "model_calls": {
    "cloud": 40,
    "local": 5
  },
  "model_seconds": 339.49,
  "near_repeat_turns": [],
  "questions_between_officers": 1,
  "sokolov_questions": 0,
  "scene_endings": {
    "1": "goal",
    "2": "goal",
    "3": "goal"
  },
  "protocol_status": "disputed",
  "protocol_rounds": 1
}


# Поправка к протоколу

Вымышленная сцена. Результат генерации моделей.

## Действие 1. Подписанная версия

*Орлов стоит у окна, спиной к столу. Волкова, раскрыв журнал, ведёт пальцем по строкам. Соколов садится напротив, кладёт ладони на столешницу.*

**Орлов:** Соколов, в записке я исходил из того, что файл ушёл из-за пределов лаборатории: вход отмечен в 21:14, а отправка — в 21:06. Скажите прямо: после ухода вы в лабораторию возвращались?

**Соколов:** Я вернулся. За зарядкой и распечаткой.

**Орлов:** Соколов, в первом объяснении вы написали, что после ухода в лабораторию не возвращались. Зачем вы это скрыли?

**Соколов** *(тише, после паузы):* Я скрыл возвращение, потому что боялся, что меня спросят об отправленном рабочем файле.

**Волкова:** Прежде чем делать вывод об отправке, надо сверить часы на проходной с серверным временем — иначе разница в минутах сама по себе ничего не доказывает.

*Соколов останавливается у стола, не отрывая взгляда от записки. Орлов медленно закрывает журнал регистрации. Волкова откладывает ручку и поворачивается к Соколову. Пауза. Соколов опускает глаза на распечатку в руке.*

## Действие 2. Сверка часов

*Орлов отходит от окна и останавливается напротив Соколова. Волкова берёт распечатку и поднимает её, не отрывая взгляда от Соколова. Соколов опускает глаза на лист в её руке.*

**Волкова** *(указывая на журнал, отчётливо):* Андрей, по акту сверки часы на проходной спешили на двенадцать минут: отметке 21:14 соответствует 21:02, то есть вход раньше отправки в 21:06. Что вы теперь снимаете из записки?

**Орлов** *(сдержанно, после паузы):* Снимаю вывод о внешней отправке: порядок отметок я истолковал неверно. Место отправки оставляю открытым.

*Орлов отходит от окна и кладёт ладонь на распечатку. Волкова поворачивает лист к себе. Соколов делает шаг вперёд, не поднимая глаз. Пауза. Волкова медленно кивает, Орлов убирает руку со стола.*

## Действие 3. Цель и разрешение

*Орлов откладывает лист, Волкова поворачивает папку к Соколову. Соколов проводит ладонью по краю стола и опускает взгляд на бумаги.*

**Орлов:** Марк просил черновую сводку по испытаниям для обсуждения консультации. Соколов, зачем вы её отправили?

**Соколов:** Я отправил черновую сводку, чтобы обсудить консультацию с Марком.

**Волкова:** По согласованию внешней передачи в материалах ничего нет — отметку об отправке я отделяю от самого действия. Соколов, вы получали разрешение на передачу этой сводки?

**Соколов:** Я не получил разрешения на передачу этой сводки.

**Волкова:** По словам Соколова, сводку он отправил для обсуждения консультации с Марком и разрешения на передачу не получал. Материалы не устанавливают ни места отправки, ни умысла.

*Волкова откладывает лист в сторону и смотрит на Соколова поверх очков. Орлов медленно закрывает папку, не отводя взгляда от Соколова.*

## Проект итогового протокола

*Формулировки и источники зачитаны всем участникам.*

### Факты и показания

- В первом объяснении Соколов отрицал возвращение. В разговоре он уточнил: «Я вернулся. За зарядкой и распечаткой».
  - `statement:3`: Соколов: «После этого в лабораторию не возвращался. Поехал домой».
  - `turn:2`: Я вернулся. За зарядкой и распечаткой.
- О цели отправки Соколов сказал: «Я отправил черновую сводку, чтобы обсудить консультацию с Марком».
  - `mail:2`: Марк: «Для обсуждения консультации пришлите черновую сводку по испытаниям. Условия оплаты обсудим отдельно».
  - `turn:9`: Я отправил черновую сводку, чтобы обсудить консультацию с Марком.
- На вопрос, получал ли он разрешение, Соколов ответил: «Я не получил разрешения на передачу этой сводки». В переписке согласование не зафиксировано.
  - `mail:5`: Согласование внешней передачи этого документа в материалах отсутствует.
  - `turn:10`: По согласованию внешней передачи в материалах ничего нет — отметку об отправке я отделяю от самого действия. Соколов, вы получали разрешение на передачу этой сводки?
  - `turn:11`: Я не получил разрешения на передачу этой сводки.
- Причину умолчания о возвращении Соколов объяснил так: «Я скрыл возвращение, потому что боялся, что меня спросят об отправленном рабочем файле».
  - `turn:4`: Я скрыл возвращение, потому что боялся, что меня спросят об отправленном рабочем файле.

### Исправлено

- Орлов снял вывод о внешней отправке. В записке отметка входа 21:14 шла после отправки в 21:06. По акту ей соответствует 21:02, то есть отметка входа теперь раньше отправки.
  - `preliminary:3`: Рабочая версия: файл отправлен в 21:06, до повторного входа в лабораторию, отмеченного в журнале в 21:14.
  - `preliminary:4`: Вывод записки: отправка из-за пределов лаборатории подтверждает заранее подготовленную внешнюю передачу.
  - `clock:4`: Запись повторного входа 21:14 соответствует серверному времени 21:02.
  - `calc:10`: {"recorded": "21:14", "offset_minutes": 12, "actual": "21:02", "note": "Расчёт не подтверждает верность исходной поправки; нужен документ."}
  - `mail:3`: Отправка с личного телефона Соколова: 21:06 по серверному времени почтового сервиса.
  - `turn:7`: Снимаю вывод о внешней отправке: порядок отметок я истолковал неверно. Место отправки оставляю открытым.

### Требует проверки

- Сверка часов не устанавливает местонахождение телефона в момент отправки.
  - `access:7`: Срабатывание пропуска само по себе не устанавливает, кто физически им пользовался и что он делал в помещении.
  - `mail:9`: Местоположение телефона в момент отправки в этой выдержке не зафиксировано.
- Личность Марка и его связи не установлены. Материалы не доказывают умысел на шпионаж.
  - `mail:7`: Личность Марка и его возможные связи с иностранными организациями не установлены.
  - `mail:8`: Материалы не дают основания считать сам факт шпионажа доказанным.

### Отзывы участников

**Орлов (возражает):** Возражение: первый источник — лишь цитата Соколова без указания, что это его первое объяснение. Второй подтверждает возвращение, но не отрицание. Формулировку о первом объяснении снимаю как недоказанную.

**Волкова (согласна):** Согласна: поправка следует из источников. Запись 21:14 по серверному времени равна 21:02, отправка в 21:06, значит вход раньше отправки. Вывод о внешней передаче снят, место отправки оставлено открытым.

**Соколов (согласен):** Согласен с изменением формулировки, так как интерпретация порядка отметок была неверной.

Статус: согласие по рассмотренным пунктам не достигнуто.


## Страница пьесы

Здесь собирается `result/<RUN_NAME>/site/index.html`. На странице можно читать пьесу по действиям, сравнивать время до и после поправки и открывать строки, на которые ссылается протокол.


In [12]:
from html import escape


PAGE_TEMPLATE = r"""<!doctype html>
<html lang="ru">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <meta name="description" content="Пьеса трёх LLM-персонажей. Сверьте хронологию, прочитайте разговор и проверьте источники итогового протокола.">
  <meta name="theme-color" content="#f4f0e7">
  <title>Поправка к протоколу · Агентский театр</title>
  <style>
:root {
  color-scheme: light;
  --paper: #f4f0e7;
  --surface: #fbf8f1;
  --ink: #202622;
  --muted: #545a53;
  --rule: #d2cdbf;
  --red: #9c3227;
  --blue: #2c5465;
  --serif: 'Playfair Display', Georgia, serif;
  --body: 'Source Sans 3', 'Segoe UI', sans-serif;
  --mono: 'IBM Plex Mono', 'Cascadia Mono', Consolas, monospace;
}

*, *::before, *::after { box-sizing: border-box; }
[hidden] { display: none !important; }
html { scroll-behavior: smooth; }
body { margin: 0; color: var(--ink); background: var(--paper); font: 1rem/1.55 var(--body); }
button, a { -webkit-tap-highlight-color: transparent; }
button { font: inherit; }
a { color: inherit; text-underline-offset: .22em; }
a:focus-visible, button:focus-visible, summary:focus-visible, [tabindex="-1"]:focus-visible { outline: 2px solid var(--blue); outline-offset: 5px; }
h1, h2, h3, p, dl, dd, ul, ol, blockquote { margin: 0; }
ol, ul { padding: 0; }
h1, h2 { font-family: var(--serif); font-weight: 500; line-height: 1.03; letter-spacing: -.045em; }
h2 { font-size: clamp(2.5rem, 6vw, 4.6rem); }
code { font-family: var(--mono); font-size: .85em; overflow-wrap: anywhere; }
[id] { scroll-margin-top: 1.4rem; }
.wrap { width: min(1160px, calc(100% - 88px)); margin-inline: auto; }
.skip-link { position: absolute; left: 1rem; top: .75rem; z-index: 5; padding: .5rem 1rem; background: var(--surface); transform: translateY(-180%); }
.skip-link:focus { transform: none; }

.masthead { min-height: 75px; border-bottom: 1px solid var(--rule); display: flex; align-items: center; justify-content: space-between; gap: 1rem; }
.masthead a { text-decoration: none; }
.brand { display: inline-flex; align-items: center; gap: .65rem; font-size: .94rem; font-weight: 750; }
.brand > span { width: 19px; height: 19px; border: 1px solid var(--red); background: repeating-linear-gradient(var(--red) 0 1px, transparent 1px 4px) center / 9px 9px no-repeat; transform: rotate(-8deg); }
.masthead nav { display: flex; flex-wrap: wrap; gap: clamp(.7rem, 3vw, 2.3rem); font-size: .89rem; color: var(--muted); }
.masthead nav a:hover, .quiet:hover, .footer a:hover { color: var(--red); }

.hero { display: grid; grid-template-columns: 1.08fr .92fr; align-items: center; gap: clamp(1.6rem, 4vw, 4rem); padding-block: 3.25rem 2.8rem; }
.label { display: flex; gap: .55rem; align-items: center; color: var(--red); font: 700 .73rem/1.35 var(--mono); text-transform: uppercase; letter-spacing: .095em; }
.signal { width: 8px; height: 8px; background: var(--red); border-radius: 50%; }
.hero h1 { font-size: clamp(3.75rem, 7.05vw, 6.65rem); margin-block: 1.15rem 1.35rem; }
.hero h1 em { color: var(--red); font-weight: inherit; }
.intro { max-width: 34ch; color: var(--muted); font-size: clamp(1.08rem, 1.65vw, 1.3rem); }
.hero-links { display: flex; align-items: center; flex-wrap: wrap; gap: 1rem 1.4rem; margin-top: 1.8rem; }
.primary, .quiet { display: inline-flex; align-items: center; justify-content: center; gap: .65rem; font-size: .9rem; font-weight: 700; }
.primary { padding: .7rem 1.05rem; background: var(--ink); color: var(--surface); border: 1px solid var(--ink); text-decoration: none; }
.primary:hover { background: var(--red); border-color: var(--red); }
.quiet { min-height: 2.5rem; }
.hero-art { isolation: isolate; display: grid; place-items: center; min-height: clamp(19rem, 35vw, 28.4rem); position: relative; overflow: hidden;
  background: radial-gradient(ellipse at 45% 37%, #496269 0, #2c4043 43%, #172629 100%); }
.hero-art::before { content: ''; position: absolute; inset: 0; opacity: .3; pointer-events: none;
  background: repeating-linear-gradient(0deg, #f6f6e70d 0 1px, transparent 1px 5px), repeating-linear-gradient(90deg, #10161738 0 1px, transparent 1px 7px); }
.folder { position: absolute; width: 77%; height: 64%; transform: rotate(-11deg) translate(-1%, 5%); background: #9f7850; border: 1px solid #b79b72;
  box-shadow: -12px 22px 35px #0b131385, inset 0 0 38px #3c2a164d; }
.folder::before { content: ''; position: absolute; top: -13px; left: 0; width: 34%; height: 14px; border-radius: 8px 10px 0 0; background: inherit; }
.exhibit { width: 73%; container-type: inline-size; position: relative; padding: 1.2rem 1.2rem 1rem; transform: rotate(7deg) translate(3%, -1%); color: #222921;
  background: linear-gradient(145deg, #f8f0dc, #e9ddbf); box-shadow: 3px 16px 25px #161e1e77; }
.exhibit::after { content: ''; width: 18px; height: 18px; position: absolute; top: 0; right: 0; background: linear-gradient(45deg, #c9b991 50%, #33494c 51%); }
.exhibit-head, .exhibit-foot { display: flex; align-items: center; justify-content: space-between; gap: .4rem; font: 700 clamp(.55rem, 1.12vw, .75rem)/1.2 var(--mono); letter-spacing: .065em; }
.exhibit-head { padding-bottom: .65rem; border-bottom: 1px solid #91856b; }
.exhibit-head span:last-child, .exhibit-foot > span { color: #57513f; }
.exhibit > p { margin-block: .9rem .55rem; font: 500 clamp(1.5rem, 3.1vw, 2.65rem)/1.02 var(--serif); letter-spacing: -.06em; }
.exhibit-times { display: flex; flex-wrap: nowrap; align-items: center; justify-content: space-between; gap: .1em; margin-block: .9rem .75rem; padding: .48rem .05rem; border-block: 1px solid #a49a83; font: clamp(1rem, 2.6vw, 2.5rem)/1.2 var(--mono); font-size: clamp(1rem, 12cqi, 2.5rem); }
.exhibit-times s { color: #615b4a; text-decoration-color: var(--red); text-decoration-thickness: .055em; }
.exhibit-times strong, .exhibit-times span { color: var(--red); }
.exhibit-times span { font-size: .72em; }
.exhibit-foot b { transform: rotate(-6deg); color: var(--red); border: 1.5px solid var(--red); padding: .32rem .45rem; white-space: nowrap; }

.stats { display: grid; grid-template-columns: repeat(4, 1fr); border-block: 1px solid var(--rule); }
.stats > div { display: flex; flex-direction: column-reverse; gap: .12rem; padding: .85rem 1.25rem .95rem; }
.stats > div:first-child { padding-inline-start: 0; }
.stats > div + div { border-left: 1px solid var(--rule); }
.stats dt { color: var(--muted); font-size: .9rem; }
.stats dd { font: 500 clamp(1.7rem, 3.3vw, 2.65rem)/1.15 var(--serif); letter-spacing: -.05em; }

.section { padding-block: clamp(3rem, 7vw, 6.2rem) 0; }
.section-head { display: flex; justify-content: space-between; align-items: end; gap: 1.5rem 2.5rem; margin-bottom: 1.45rem; }
.section-head .label { margin-bottom: .65rem; }
.section-head > p { max-width: 42ch; color: var(--muted); }
.timeline { --entrance: 87.5%; --dispatch: 37.5%; background: #e7eded; border: 1px solid #c3cfce; padding: clamp(.85rem, 2.8vw, 1.75rem); }
.timeline.after { --entrance: 12.5%; }
.timeline-top, .timeline-bottom { display: flex; flex-wrap: wrap; justify-content: space-between; align-items: center; gap: .65rem 1rem; }
.timeline-top:has(.segmented[hidden]) { display: none; }
.segmented { display: inline-flex; flex-wrap: wrap; gap: 2px; border: 1px solid #b5c4c3; padding: 3px; }
.segmented button { border: 0; color: #32484c; background: transparent; font-weight: 650; cursor: pointer; padding: .38rem .65rem; }
.segmented button:hover { background: #dae2e2; }
.segmented button[aria-pressed="true"] { background: #284650; color: white; }
.offset { border: 1px solid #b99b94; color: var(--red); padding: .27rem .6rem; font: 700 .8rem var(--mono); }
.chart { display: grid; grid-template-columns: max-content minmax(0, 1fr); gap: .55rem clamp(.75rem, 4vw, 2.4rem); align-items: center; margin: 1.3rem 1rem 1.15rem 0; }
.track-label { font-size: .88rem; font-weight: 650; display: flex; gap: .5rem; align-items: center; }
.track-label i { display: block; width: .55rem; height: .55rem; flex-shrink: 0; background: currentColor; border-radius: 50%; }
.access-label { color: var(--red); }
.mail-label { color: var(--blue); }
.track { position: relative; height: 2.65rem; background: repeating-linear-gradient(90deg, #a4b4b5 0 1px, transparent 1px 25%); border-right: 1px solid #a4b4b5; }
.track::before { content: ''; position: absolute; inset: 50% 0 auto; height: 1px; background: #9aacad; }
.marker { position: absolute; inset: 0 auto 0 0; left: var(--entrance); display: grid; place-items: center; transform: translateX(-50%); color: var(--red); transition: left .55s cubic-bezier(.25,.75,.3,1); }
.marker.dispatch { left: var(--dispatch); color: var(--blue); }
.marker b { color: #fff; background: var(--red); border-radius: 3px; font: 700 clamp(.75rem, 2vw, .98rem)/1.2 var(--mono); padding: .32rem .44rem; }
.dispatch b { background: var(--blue); }
.ticks { display: flex; justify-content: space-between; color: #4d6264; font: clamp(.64rem, 1.5vw, .77rem) var(--mono); margin-inline: -1.5ch; }
.timeline-bottom { border-top: 1px solid #bdcccb; padding-top: .85rem; }
.timeline-bottom p { font-weight: 650; }
.references { display: flex; flex-wrap: wrap; gap: .35rem; align-items: center; }
.source-link { display: inline-block; padding: .17rem .4rem; border: 1px solid #b8c6c4; color: #2b5059; background: #f6faf84a; font: .74rem/1.45 var(--mono); text-decoration: none; }
.source-link:hover { background: #dbe7e6; border-color: #839d9f; }
.annotation { max-width: 73ch; color: var(--muted); font-size: .93rem; margin-top: .9rem; }

.orlov { --speaker: #3f5266; --badge: #e1e8f0; }
.volkova { --speaker: #2f5b50; --badge: #e1ebe4; }
.sokolov { --speaker: #904334; --badge: #f0e1d8; }
.play-grid { display: grid; grid-template-columns: minmax(12.5rem, .29fr) minmax(0, 1fr); gap: 1.1rem; align-items: start; }
.cast { border-top: 3px solid var(--ink); padding: 1.1rem 1rem; background: #ebe7dd; }
.cast h3, .reviews h3, .method-grid h3 { font-size: 1rem; }
.cast ul { list-style: none; margin-block: 1rem; }
.cast li { display: flex; gap: .6rem; align-items: start; padding-block: .65rem; border-top: 1px solid var(--rule); }
.cast small { display: block; color: var(--muted); }
.cast p { color: var(--muted); font-size: .78rem; }
.initial { background: var(--badge); color: var(--speaker); flex: 0 0 1.9rem; height: 1.9rem; display: grid; place-items: center; font: 700 .8rem var(--mono); }
.script { border: 1px solid var(--rule); background: var(--surface); }
.act-nav { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); border-bottom: 1px solid var(--rule); }
.act-nav a { display: flex; align-items: baseline; gap: .45rem; padding: .8rem .65rem; border-bottom: 2px solid transparent; text-decoration: none; font-size: .84rem; font-weight: 650; }
.act-nav a + a { border-left: 1px solid var(--rule); }
.act-nav a > span { color: var(--red); font: .8rem var(--mono); }
.act-nav a[aria-selected="true"] { border-bottom-color: var(--red); background: #f4e9df; }
.act-nav a:hover { background: #eeeae0; }
.act { padding: clamp(1rem, 3.4vw, 2rem); }
.act + .act { border-top: 1px solid var(--rule); }
.act h3 { font: 500 clamp(1.3rem, 4vw, 2rem)/1.18 var(--serif); }
.act audio { display: block; width: min(100%, 28rem); margin-top: .8rem; }
.direction { font-family: var(--serif); font-style: italic; color: var(--muted); padding-block: .85rem; }
.speeches { list-style: none; }
.speech { display: grid; grid-template-columns: 1.8rem minmax(0, 1fr); gap: .6rem; align-items: start; border-top: 1px solid #e3ded2; padding: .67rem 0; }
.speech:last-child { border-bottom: 1px solid #e3ded2; }
.turn-number { color: #646760; font: .73rem/1.65 var(--mono); }
.speech b, .reviews b { color: var(--speaker); }
.speech.is-speaking { background: var(--badge); box-shadow: -.45rem 0 var(--badge), .45rem 0 var(--badge); }
.speech p { line-height: 1.55; }
.delivery { color: var(--muted); font-size: .9em; }
.speech:target, .document li:target, .protocol li:target { background: #f3e9cf; }

.protocol-grid { display: grid; grid-template-columns: minmax(0, 1.55fr) minmax(15rem, 1fr); gap: 1.1rem; align-items: start; }
.protocol-group, .document { background: var(--surface); border: 1px solid var(--rule); margin-bottom: .55rem; }
summary { display: flex; align-items: center; gap: .65rem; cursor: pointer; font-weight: 700; padding: .85rem 1rem; }
summary::marker, summary::-webkit-details-marker { display: none; content: ''; }
summary::before { content: '+'; flex: 0 0 1rem; color: var(--red); font: 700 1rem var(--mono); }
details[open] > summary::before { content: '−'; }
summary:hover { background: #f0ebe0; }
.count { margin-left: auto; color: var(--muted); font: .75rem var(--mono); }
.protocol-group ol { border-top: 1px solid var(--rule); padding: .05rem 1rem .1rem 2.15rem; }
.protocol-group li { padding: .6rem .1rem; }
.protocol-group li + li { border-top: 1px solid #e3ded2; }
.protocol-group .references { margin-top: .45rem; }
.reviews { background: #ebe7dd; padding: 1rem; }
.reviews ul { list-style: none; }
.reviews li { padding: .7rem 0; border-top: 1px solid var(--rule); margin-top: .35rem; }
.reviews p { font-size: .91rem; }
.reviews a { font-size: .82rem; color: var(--blue); }
.review-status { margin-left: .45rem; padding: .11rem .36rem; font: .69rem var(--mono); background: var(--badge); color: var(--speaker); }
.earlier-review { border-top: 1px solid var(--rule); margin-top: .35rem; }
.earlier-review summary { padding-inline: 0; font-size: .88rem; }
.earlier-review li { margin-top: 0; }
.documents { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: .65rem; align-items: start; }
.document { margin: 0; }
.document ol { list-style: none; border-top: 1px solid var(--rule); padding: .15rem 1rem .35rem; }
.document li { display: grid; grid-template-columns: minmax(4.8rem, max-content) minmax(0, 1fr); padding: .35rem 0; gap: .1rem .6rem; }
.document li + li { border-top: 1px solid #e3ded2; }
.document code { color: var(--blue); }
.document li a { grid-column: 2; font-size: .83rem; }
.method-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: .85rem 1.3rem; }
.method-grid article { padding-top: .8rem; border-top: 2px solid var(--red); }
.method-grid p { margin-top: .35rem; color: var(--muted); }
.note { max-width: 72ch; margin-top: 1.2rem; padding: .65rem .85rem; border-left: 2px solid var(--red); background: #efe7da; }
.footer { display: flex; justify-content: space-between; flex-wrap: wrap; gap: .45rem 1rem; border-top: 1px solid var(--rule); padding-block: 1rem 1.4rem; margin-top: 3.8rem; color: var(--muted); font-size: .86rem; }

.source-dialog { width: min(34rem, calc(100% - 2rem)); padding: 1.5rem; border: 1px solid var(--rule); background: var(--surface); color: var(--ink); }
.source-dialog::backdrop { background: #192622a6; }
.source-dialog h2 { font-size: 2rem; padding-right: 2rem; }
.source-dialog blockquote { padding-left: .8rem; margin-block: 1rem; border-left: 2px solid var(--red); }
.close { display: grid; place-items: center; position: absolute; top: .8rem; right: .8rem; background: transparent; border: 1px solid var(--rule); color: var(--ink); width: 2rem; height: 2rem; font-size: 1.3rem; cursor: pointer; }
.close:hover { background: #e9e1d4; }

@media (max-width: 880px) {
  .wrap { width: calc(100% - 54px); }
  .hero { grid-template-columns: 1.02fr .98fr; }
  .hero h1 { font-size: clamp(3.1rem, 7.9vw, 5rem); }
  .play-grid { grid-template-columns: minmax(11rem, .3fr) minmax(0, 1fr); }
}
@media (max-width: 640px) {
  .wrap { width: calc(100% - 32px); }
  .masthead { min-height: auto; padding-block: .8rem; align-items: start; flex-direction: column; gap: .55rem; }
  .masthead nav { gap: .9rem; }
  .hero { grid-template-columns: 1fr; padding-block: 2.25rem 1.55rem; gap: 1.5rem; }
  .hero h1 { font-size: clamp(3.1rem, 11vw, 4.6rem); }
  .hero-links { margin-top: 1.1rem; }
  .hero-art { min-height: 17rem; }
  .exhibit { width: min(74%, 21rem); }
  .stats { grid-template-columns: repeat(2, 1fr); }
  .stats > div:nth-child(3) { border-left: 0; padding-inline-start: 0; }
  .stats > div:nth-child(n+3) { border-top: 1px solid var(--rule); }
  .section-head { align-items: start; flex-direction: column; gap: .65rem; }
  .section-head .quiet { min-height: auto; }
  .chart { grid-template-columns: minmax(0, 1fr); margin-inline: .42rem; gap: .35rem; }
  .track-label { padding-top: .2rem; }
  .chart > div:empty { display: none; }
  .play-grid, .protocol-grid, .documents, .method-grid { grid-template-columns: minmax(0, 1fr); }
  .cast ul { margin-block: .45rem; }
  .act-nav a { flex-direction: column; gap: .2rem; padding: .6rem .45rem; }
  .document li { grid-template-columns: minmax(0, 1fr); }
  .document li a { grid-column: 1; }
  .footer { margin-top: 2.5rem; }
}
@media (prefers-reduced-motion: reduce) {
  html { scroll-behavior: auto; }
  .marker { transition: none; }
}
  </style>
</head>
<body>
  <a class="skip-link" href="#main">К содержанию</a>
  <header class="masthead wrap">
    <a class="brand" href="#main"><span aria-hidden="true"></span> Агентский театр</a>
    <nav aria-label="Разделы">
      <a href="#timeline">Хронология</a><a href="#play">Пьеса</a><a href="#protocol">Протокол</a>
    </nav>
  </header>
  <main id="main">
    <section class="hero wrap" aria-labelledby="title">
      <div class="hero-copy">
        <p class="label"><span class="signal"></span> Дело «Север»</p>
        <h1 id="title">Поправка<br><em>к протоколу</em></h1>
        <p class="intro">Орлов уже подписал вывод об отправке рабочего файла. Волкова находит расхождение в часах проходной, а Соколов меняет показания.</p>
        <div class="hero-links"><a class="primary" href="#play">Читать пьесу <span aria-hidden="true">↗</span></a><a class="quiet" href="#timeline">Сверить время <span aria-hidden="true">↓</span></a></div>
      </div>
      <div class="hero-art" aria-hidden="true">
        <div class="folder"></div>
        <div class="exhibit">
          <div class="exhibit-head"><span>АКТ СВЕРКИ</span><span>ДЕЛО «СЕВЕР»</span></div>
          <p>Повторный вход<br>по пропуску</p>
          <div class="exhibit-times"><s>21:14</s><span>↘</span><strong>21:02</strong></div>
          <div class="exhibit-foot"><span>ВРЕМЯ СЕРВЕРА</span><b>−12 МИН</b></div>
        </div>
      </div>
    </section>

    <dl class="stats wrap" aria-label="Сохранённый прогон">
      <div><dt>Действия</dt><dd>{{ACTS}}</dd></div><div><dt>Реплики</dt><dd>{{TURNS}}</dd></div>
      <div><dt>Вызовы инструментов</dt><dd>{{TOOL_CALLS}}</dd></div><div><dt>Редакции протокола</dt><dd>{{ROUNDS}}</dd></div>
    </dl>

    <section id="timeline" class="section wrap" aria-labelledby="timeline-title">
      <div class="section-head"><div><p class="label">Хронология дела</p><h2 id="timeline-title">Двенадцать минут</h2></div>
        <p>Контроллер проходной спешил. Примените поправку к записи о входе и сравните её с серверным временем отправки.</p></div>
      <div class="timeline" data-timeline>
        <div class="timeline-top">
          <div class="segmented" aria-label="Состояние хронологии" hidden>
            <button type="button" aria-pressed="true" data-time="before">По записке</button><button type="button" aria-pressed="false" data-time="after">После сверки</button>
          </div>
          <span class="offset" data-offset hidden>−12 мин</span>
        </div>
        <div class="chart" aria-hidden="true">
          <div class="track-label access-label"><i></i> Вход по пропуску</div>
          <div class="track"><span class="marker entrance"><b data-entry>21:14</b></span></div>
          <div class="track-label mail-label"><i></i> Отправка файла</div>
          <div class="track"><span class="marker dispatch"><b>21:06</b></span></div>
          <div></div><div class="ticks"><span>21:00</span><span>21:04</span><span>21:08</span><span>21:12</span><span>21:16</span></div>
        </div>
        <div class="timeline-bottom"><p data-order aria-live="polite">По записке вход отмечен в 21:14, после отправки в 21:06.</p>
          <div class="references"><a class="source-link" href="#source-clock-4" data-source>Акт сверки ↗</a><a class="source-link" href="#source-mail-3" data-source>Переписка ↗</a></div></div>
      </div>
      <p class="annotation">После поправки отметка входа сдвигается на 21:02. Сверка меняет порядок записей. Где был телефон, по этим записям определить нельзя.</p>
    </section>

    <section id="play" class="section wrap" aria-labelledby="play-title">
      <div class="section-head"><h2 id="play-title">Пьеса</h2></div>
      <div class="play-grid">
        <aside class="cast"><h3>Действующие лица</h3><ul>{{CAST}}</ul><p>Персонажи и материалы дела вымышлены.</p></aside>
        <div class="script"><nav class="act-nav" aria-label="Действия пьесы">{{TABS}}</nav>{{SCENES}}</div>
      </div>
    </section>

    <section id="protocol" class="section wrap" aria-labelledby="protocol-title">
      <div class="section-head"><div><p class="label">Итог разговора</p><h2 id="protocol-title">Протокол</h2></div><p>{{STATUS}} Номера под пунктами ведут к строкам документов, репликам или расчётам.</p></div>
      <div class="protocol-grid"><div class="protocol">{{PROTOCOL}}</div>
        <div class="reviews"><h3>Отзывы участников</h3><ul>{{REVIEWS}}</ul>{{PREVIOUS}}</div></div>
    </section>

    <section id="sources" class="section wrap" aria-labelledby="source-title">
      <div class="section-head"><div><p class="label">Строки и ссылки</p><h2 id="source-title">Материалы дела</h2></div><p>Документы становились доступны постепенно. Номер действия указан рядом с каждым документом.</p></div>
      <div class="documents">{{SOURCES}}</div>
    </section>

    <section id="method" class="section wrap" aria-labelledby="method-title">
      <div class="section-head"><h2 id="method-title">Как получилась пьеса</h2></div>
      <div class="method-grid">
        <article><h3>План и очередь</h3><p>Облачная модель составляет план из трёх действий. На прямой вопрос отвечает адресат. Порядок эпизодов берётся из плана, составленного до диалога.</p></article>
        <article><h3>Две модели</h3><p>Орлова и Волкову играет <code>{{CLOUD}}</code> через Yandex AI Studio. Соколова играет локальная <code>{{LOCAL}}</code>.</p></article>
        <article><h3>Источники</h3><p>Сотрудники ищут строки в материалах, читают выдержки и проверяют поправку часов. При составлении протокола ссылки сверяются с журналом.</p></article>
      </div>
    </section>
  </main>
  <footer class="footer wrap"><span>Агентский театр · Лабораторная работа{{AUDIO_CREDIT}}</span><a href="https://github.com/MAILabs-Education/Agent_Theatre">Условие задания ↗</a></footer>

  <dialog class="source-dialog" aria-labelledby="source-dialog-title">
    <form method="dialog"><button class="close" aria-label="Закрыть">×</button></form>
    <h2 id="source-dialog-title">Источник</h2>
    <blockquote></blockquote>
    <a class="quiet" href="#sources">Перейти к источнику <span aria-hidden="true">↗</span></a>
  </dialog>
  <script>
const timeline = document.querySelector('[data-timeline]');
const timeButtons = [...timeline.querySelectorAll('[data-time]')];
timeline.querySelector('.segmented').hidden = false;

for (const button of timeButtons) {
  button.addEventListener('click', () => {
    const corrected = button.dataset.time === 'after';
    timeline.classList.toggle('after', corrected);
    timeline.querySelector('[data-entry]').textContent = corrected ? '21:02' : '21:14';
    timeline.querySelector('[data-offset]').hidden = !corrected;
    timeline.querySelector('[data-order]').textContent = corrected
      ? 'После сверки вход отмечен в 21:02, за четыре минуты до отправки.'
      : 'По записке вход отмечен в 21:14, после отправки в 21:06.';
    for (const option of timeButtons) option.setAttribute('aria-pressed', String(option === button));
  });
}

const tabList = document.querySelector('.act-nav');
const tabs = [...tabList.querySelectorAll('a')];
const panels = tabs.map(tab => document.getElementById(tab.hash.slice(1)));

function selectAct(index) {
  for (const [i, tab] of tabs.entries()) {
    const active = i === index;
    tab.setAttribute('aria-selected', String(active));
    tab.tabIndex = active ? 0 : -1;
    panels[i].hidden = !active;
    if (!active) panels[i].querySelector('audio')?.pause();
  }
}

function openAnchor(hash) {
  const target = document.getElementById(hash.slice(1));
  if (!target) return null;
  const panel = target.closest('.act');
  if (panel) selectAct(panels.indexOf(panel));
  const details = target.closest('details');
  if (details) details.open = true;
  return target;
}

tabList.setAttribute('role', 'tablist');
for (const [i, tab] of tabs.entries()) {
  tab.setAttribute('role', 'tab');
  tab.setAttribute('aria-controls', panels[i].id);
  panels[i].setAttribute('role', 'tabpanel');
  panels[i].setAttribute('aria-labelledby', tab.id);
  panels[i].tabIndex = 0;
  tab.addEventListener('click', event => {
    if (event.ctrlKey || event.metaKey || event.shiftKey || event.altKey) return;
    event.preventDefault();
    selectAct(i);
    history.replaceState(null, '', tab.hash);
  });
  tab.addEventListener('keydown', event => {
    const next = {
      ArrowLeft: (i + tabs.length - 1) % tabs.length,
      ArrowRight: (i + 1) % tabs.length,
      Home: 0,
      End: tabs.length - 1,
    }[event.key];
    if (event.key === ' ') {
      event.preventDefault();
      tab.click();
    } else if (next !== undefined) {
      event.preventDefault();
      tabs[next].click();
      tabs[next].focus();
    }
  });
}

selectAct(0);
openAnchor(location.hash);
window.addEventListener('hashchange', () => openAnchor(location.hash)?.scrollIntoView());

for (const panel of panels) {
  const player = panel.querySelector('audio');
  if (!player) continue;
  const cues = [...panel.querySelectorAll('[data-start]')].map(row => ({
    row, start: Number(row.dataset.start), end: Number(row.dataset.end),
  }));
  let activeCue;
  function syncSpeech() {
    const time = player.currentTime;
    const cue = cues.find(({start, end}) => start <= time && time < end);
    if (cue === activeCue) return;
    activeCue?.row.classList.remove('is-speaking');
    cue?.row.classList.add('is-speaking');
    activeCue = cue;
  }
  player.addEventListener('timeupdate', syncSpeech);
  player.addEventListener('seeked', syncSpeech);
}

const dialog = document.querySelector('.source-dialog');
const sourceTarget = dialog.querySelector('a');
document.addEventListener('click', event => {
  const link = event.target.closest('a[href^="#"]');
  if (!link || event.defaultPrevented || event.ctrlKey || event.metaKey || event.shiftKey || event.altKey) return;
  const target = document.getElementById(link.hash.slice(1));
  if (link.hasAttribute('data-source') && typeof dialog.showModal === 'function' && target) {
    event.preventDefault();
    dialog.querySelector('h2').textContent = target.querySelector('code').textContent;
    dialog.querySelector('blockquote').textContent = target.querySelector('p').textContent;
    sourceTarget.setAttribute('href', link.hash);
    dialog.showModal();
    return;
  }
  if (link === sourceTarget) dialog.close();
  openAnchor(link.hash);
});
  </script>
</body>
</html>
"""


ROLES = {
    "orlov": ("Орлов", "Автор предварительной записки", "О"),
    "volkova": ("Волкова", "Проверяет документы и хронологию", "В"),
    "sokolov": ("Соколов", "Инженер, отправивший сводку", "С"),
}
SECTIONS = {"established": "Факты и показания", "corrected": "Исправлено", "unresolved": "Требует проверки"}


def source_id(ref):
    return "source-" + ref.replace(":", "-")


def source_link(ref):
    return f'<a class="source-link" href="#{escape(source_id(ref))}" data-source>{escape(ref)}</a>'


def render_cast():
    return "\n".join(
        f'<li class="{role}"><span class="initial" aria-hidden="true">{initial}</span>'
        f'<span><strong>{name}</strong><small>{description}</small></span></li>'
        for role, (name, description, initial) in ROLES.items()
    )


def render_play(run, tracks):
    indexed = list(enumerate(run["turns"], 1))
    tabs, panels = [], []
    for act in run["plan"]["acts"]:
        number, title = act["id"], escape(act["title"])
        numeral = ("I", "II", "III")[number - 1]
        tabs.append(f'<a href="#act-{number}" id="tab-{number}"><span>{numeral}</span> {title}</a>')
        track = tracks.get(str(number))
        player = (f'<audio controls preload="metadata" src="audio/act-{number}.wav" '
                  f'aria-label="Озвучка действия {numeral}">'
                  f'<a href="audio/act-{number}.wav">Скачать озвучку</a></audio>' if track else "")
        turns = []
        for index, turn in indexed:
            if turn["act"] == number:
                role = turn["speaker"]
                name = ROLES[role][0]
                cue = track["cues"].get(str(index)) if track else None
                time = f' data-start="{cue[0]:.3f}" data-end="{cue[1]:.3f}"' if cue else ""
                delivery = delivery_cue(turn)
                note = f' <i class="delivery">({escape(delivery[0])})</i>' if delivery else ""
                turns.append(f'<li id="turn-{index}" class="speech {role}"{time}>'
                             f'<span class="turn-number" aria-hidden="true">{index:02d}</span>'
                             f'<div><b>{name}</b>{note}<p>{escape(turn["text"])}</p></div></li>')
        direction = run["directions"][str(number)]
        panels.append(f'<article class="act" id="act-{number}" aria-labelledby="heading-{number}">'
                      f'<h3 id="heading-{number}">{numeral}. {title}</h3>{player}'
                      f'<p class="direction">{escape(direction["opening"])}</p>'
                      f'<ol class="speeches">{"".join(turns)}</ol>'
                      f'<p class="direction">{escape(direction["closing"])}</p></article>')
    return "\n".join(tabs), "\n".join(panels)


def render_protocol(protocol):
    final = protocol["rounds"][-1]
    groups = []
    for section, title in SECTIONS.items():
        rows = []
        for index, row in enumerate(final["draft"][section], 1):
            refs = " ".join(map(source_link, row["sources"]))
            rows.append(f'<li id="{section}-{index}"><p>{escape(row["statement"])}</p>'
                        f'<div class="references" aria-label="Источники">{refs}</div></li>')
        count = len(rows)
        groups.append(f'<details class="protocol-group {section}"{" open" if section == "corrected" else ""}>'
                      f'<summary><span>{title}</span><span class="count">{count}</span></summary><ol>{"".join(rows)}</ol></details>')
    reviews = []
    for review in final["reviews"]:
        section, index = review["item"].split(":")
        status = {"accept": "Без возражений", "object": "Возражает", "abstain": "Воздержался"}[review["status"]]
        reviews.append(f'<li class="{review["speaker"]}"><b>{ROLES[review["speaker"]][0]}</b>'
                       f'<span class="review-status">{status}</span><p>{escape(review["text"])}</p>'
                       f'<a href="#{section}-{index}">{SECTIONS[section]}, п. {index}</a></li>')
    objections = [f'<li class="{review["speaker"]}"><b>{ROLES[review["speaker"]][0]}</b><p>{escape(review["text"])}</p></li>'
                  for revision in protocol["rounds"][:-1] for review in revision["reviews"] if review["status"] == "object"]
    previous = ('<details class="earlier-review"><summary>Возражения к первой редакции</summary>'
                f'<ul>{"".join(objections)}</ul></details>' if objections else "")
    return "\n".join(groups), "\n".join(reviews), previous


def render_sources(run):
    groups = []
    for key, document in sorted(DOCUMENTS.items(), key=lambda pair: pair[1]["act"]):
        lines = CASE_TEXTS[key].splitlines()
        rows = "".join(
            f'<li id="{source_id(f"{key}:{i}")}"><code>{key}:{i}</code><p>{escape(line)}</p></li>'
            for i, line in enumerate(lines, 1)
        )
        groups.append(f'<details class="document"><summary><span>{escape(document["title"])}</span>'
                      f'<span class="count">Акт {document["act"]}</span></summary><ol>{rows}</ol></details>')
    extra = []
    for ref, row in run["protocol"]["sources"].items():
        if row["kind"] == "testimony":
            number = int(ref.split(":")[1])
            extra.append(f'<li id="{source_id(ref)}"><code>{ref}</code><p>{escape(row["speaker"])}: {escape(row["text"])}</p>'
                         f'<a href="#turn-{number}">К реплике</a></li>')
        elif row["kind"] == "calculation":
            calc = json.loads(row["text"])
            extra.append(f'<li id="{source_id(ref)}"><code>{ref}</code><p>{escape(calc["recorded"])} − '
                         f'{calc["offset_minutes"]} мин = {escape(calc["actual"])}</p></li>')
    groups.append(f'<details class="document"><summary>Реплики и расчёты</summary><ol>{"".join(extra)}</ol></details>')
    return "\n".join(groups)


def build_site(run, site_directory, tracks):
    tabs, scenes = render_play(run, tracks)
    protocol, reviews, previous = render_protocol(run["protocol"])
    metrics = run["metrics"]
    status = ("К рассмотренным пунктам возражений нет."
              if run["protocol"]["status"] == "agreed" else "Согласие по рассмотренным пунктам не достигнуто.")
    if run["protocol"].get("revision_error"):
        status += " Вторую редакцию завершить не удалось, поэтому сохранена первая."
    values = {
        "TABS": tabs, "SCENES": scenes, "CAST": render_cast(), "PROTOCOL": protocol, "REVIEWS": reviews, "PREVIOUS": previous,
        "SOURCES": render_sources(run), "ACTS": len(run["plan"]["acts"]), "TURNS": metrics["turns"],
        "TOOL_CALLS": sum(metrics["tools"].values()), "ROUNDS": metrics["protocol_rounds"], "STATUS": status,
        "CLOUD": escape(run["models"]["cloud"]), "LOCAL": escape(run["models"]["local"]),
        "AUDIO_CREDIT": (' · <a href="https://github.com/snakers4/silero-models#licence">Озвучка Silero</a>'
                         if tracks else ""),
    }
    page = re.sub(r"\{\{([A-Z_]+)\}\}", lambda match: str(values[match[1]]), PAGE_TEMPLATE)
    site_directory.mkdir(parents=True, exist_ok=True)
    path = site_directory / "index.html"
    path.write_text(page, encoding="utf-8", newline="\n")
    return path

In [13]:
from IPython.display import FileLink, display

site_directory = OUTPUT / "site"

## Озвучка

Silero V5 озвучивает реплики и ремарки. Рассказчик читает пометки о подаче перед выбранными репликами. У трёх персонажей и рассказчика разные голоса. Для каждого действия получается отдельный WAV-файл. Начало и конец реплик сохраняются в секундах — по ним страница подсвечивает текст во время воспроизведения.

Если рядом с блокнотом есть `REPORT.md`, после озвучки в разделе «Результаты» обновятся статистика, фрагмент диалога, отзывы и длительность дорожек.


In [14]:
import shutil
from tempfile import TemporaryDirectory
import wave


MODEL = "v5_6_ru"
MODEL_URL = f"https://models.silero.ai/models/tts/ru/{MODEL}.pt"
SAMPLE_RATE = 24000
VOICES = {"orlov": "eugene", "volkova": "xenia", "sokolov": "aidar", "narrator": "kseniya"}
NUMBERS = ("ноль", "один", "два", "три", "четыре", "пять", "шесть", "семь", "восемь", "девять",
           "десять", "одиннадцать", "двенадцать", "тринадцать", "четырнадцать", "пятнадцать",
           "шестнадцать", "семнадцать", "восемнадцать", "девятнадцать")
TENS = ("двадцать", "тридцать", "сорок", "пятьдесят")


def spoken_number(number, feminine=False):
    tens, units = divmod(number, 10)
    last = {1: "одна", 2: "две"}.get(units, NUMBERS[units]) if feminine else NUMBERS[units]
    if number < 20:
        return last if number < 10 else NUMBERS[number]
    return TENS[tens - 2] + (" " + last if units else "")


def inflect(number, one, few, many):
    if 11 <= number % 100 <= 14:
        return many
    return {1: one, 2: few, 3: few, 4: few}.get(number % 10, many)


def speak_time(match):
    hours, minutes = map(int, match.groups())
    hour = spoken_number(hours) + " " + inflect(hours, "час", "часа", "часов")
    if not minutes:
        return hour + " ровно"
    return hour + " " + spoken_number(minutes, feminine=True) + " " + inflect(minutes, "минута", "минуты", "минут")


def synthesize(run, site_directory):
    import torch

    cache = MODEL_CACHE / "silero"
    cache.mkdir(parents=True, exist_ok=True)
    weights = cache / (MODEL + ".pt")
    if not weights.exists():
        torch.hub.download_url_to_file(MODEL_URL, weights)
    torch.set_num_threads(2)
    model = torch.package.PackageImporter(weights).load_pickle("tts_models", "model")
    model.to(torch.device("cpu"))

    def say(text, role):
        # Время передаём словами, чтобы синтезатор не читал двоеточие и цифры отдельно.
        text = re.sub(r"\b([01]?\d|2[0-3]):([0-5]\d)\b", speak_time, text)
        with torch.inference_mode():
            samples = model.apply_tts(text=text, speaker=VOICES[role], sample_rate=SAMPLE_RATE)
        return samples.clamp(-1, 1).mul(30000).to(torch.int16).cpu().numpy().astype("<i2", copy=False).tobytes()

    target = site_directory / "audio"
    target.mkdir(parents=True, exist_ok=True)
    tracks = {}
    indexed = list(enumerate(run["turns"], 1))
    with TemporaryDirectory(dir=target) as folder:
        stage = Path(folder)
        for act in run["plan"]["acts"]:
            act_id = act["id"]
            file = stage / f"act-{act_id}.wav"
            cues = {}
            direction = run["directions"][str(act_id)]
            with wave.open(str(file), "wb") as track:
                track.setparams((1, 2, SAMPLE_RATE, 0, "NONE", "not compressed"))
                track.writeframes(say(direction["opening"], "narrator"))
                track.writeframes(bytes(SAMPLE_RATE))
                for number, turn in indexed:
                    if turn["act"] != act_id:
                        continue
                    delivery = delivery_cue(turn)
                    if delivery:
                        track.writeframes(say(delivery[1], "narrator"))
                        track.writeframes(bytes(SAMPLE_RATE))
                    # Подсветка начинается после ремарки, вместе с голосом персонажа.
                    start = track.getnframes() / SAMPLE_RATE
                    track.writeframes(say(turn["text"], turn["speaker"]))
                    cues[str(number)] = [round(start, 3), round(track.getnframes() / SAMPLE_RATE, 3)]
                    track.writeframes(bytes(SAMPLE_RATE * 2 // 3))
                track.writeframes(say(direction["closing"], "narrator"))
                duration = round(track.getnframes() / SAMPLE_RATE, 3)
            tracks[str(act_id)] = {"seconds": duration, "cues": cues}
            print(f"Действие {act_id}: {duration:.1f} с.", flush=True)
        # Старые дорожки заменяются только после синтеза всех трёх действий.
        for file in stage.glob("*.wav"):
            temporary = target / (file.name + ".tmp")
            shutil.copyfile(file, temporary)
            temporary.replace(target / file.name)
    return tracks


def result_section(run, tracks):
    turns = run["turns"]
    metrics = run["metrics"]
    roles = ", ".join(f"{NAMES[role]} — {metrics['roles'].get(role, 0)}" for role in NAMES)
    tools = metrics["tools"]
    choices = metrics["speaker_selection"]
    local_turns = [turn for turn in turns if turn["speaker"] == "sokolov"]
    constrained = sum(turn["backend"] == "local_constrained" for turn in local_turns)
    lines = [
        f"Действий — {len(run['plan']['acts'])}, реплик — {len(turns)}. {roles}.",
        "",
        f"Орлов и Волкова работали через `{run['models']['cloud']}`. "
        f"Реплики Соколова сформулированы локальной моделью `{run['models']['local']}`. "
        f"Ответов с ограниченным выбором фразы — {constrained}.",
        "",
        f"Ходов по плану — {choices.get('planner', 0)}, ответов по правилу адресата — {choices.get('rule', 0)}. "
        f"Поисков по файлам — {tools.get('file_search', 0)}, "
        f"чтений документов — {tools.get('read_excerpt', 0)}, "
        f"расчётов поправки — {tools.get('timeline_check', 0)}.",
    ]
    if run.get("edit_local"):
        edited = sum(bool(turn.get("edit_applied")) for turn in local_turns)
        lines += ["", f"Облачный редактор изменил {edited} реплик Соколова. Исходные ответы сохранены в журнале."]
    unfinished = [key for key, scene in run["scene_checks"].items() if scene["ending"] == "limit"]
    if unfinished:
        lines += ["", "Проверка не подтвердила достижение целей действий " + ", ".join(unfinished) + "."]
    planned = next((act["id"] for act in run["plan"]["acts"] if "concealment" in act.get("topics", [])), None)
    if planned is not None:
        order = {"purpose": "цель отправки", "permission": "разрешение на передачу",
                 "concealment": "причина умолчания"}
        subjects = [order[topic] for topic in run["plan"]["acts"][-1]["topics"] if topic in order]
        lines += ["", f"По плану Соколов объяснил умолчание в действии {planned}. "
                      f"Порядок вопросов в третьем действии: {', '.join(subjects)}."]
    for first, second in zip(turns, turns[1:]):
        if (first["act"] == second["act"] == 2 and first["speaker"] == "volkova"
                and second["speaker"] == "orlov" and "21:02" in first["text"]):
            lines += ["", "Сверка часов и ответ Орлова", "",
                      f"> **Волкова.** {first['text']}", ">",
                      f"> **Орлов.** {second['text']}"]
            break

    protocol = run["protocol"]
    reviews = protocol["rounds"][-1]["reviews"]
    statuses = {"accept": "согласие", "object": "возражение", "abstain": "воздержание"}
    votes = ", ".join(f"{NAMES[row['speaker']]} — {statuses[row['status']]}" for row in reviews)
    lines += ["", f"Редакций протокола — {len(protocol['rounds'])}. "
              f"Отзывы по выбранным пунктам: {votes}."]
    if any(row["status"] == "abstain" for row in reviews):
        lines.append("Воздержание означает, что ни один из черновиков отзыва не прошёл проверку.")
    if protocol.get("revision_error"):
        lines.append("Вторую редакцию завершить не удалось. Сохранена первая редакция.")

    duration = round(sum(track["seconds"] for track in tracks.values()))
    link = (OUTPUT / "site/index.html").relative_to(ROOT).as_posix()
    count = sum(delivery_cue(turn) is not None for turn in turns)
    lines += ["", f"Silero озвучил пьесу четырьмя голосами. Рассказчик читает ремарки в начале и конце действий "
              f"и пометки перед ключевыми репликами. Пометок к репликам — {count}. "
              f"Дорожек — {len(tracks)}, общая длительность — {duration // 60}:{duration % 60:02d}. "
              f"[Пьеса с озвучкой]({link})."]
    return "\n".join(lines)


def update_report(run, tracks):
    path = ROOT / "REPORT.md"
    if not path.is_file():
        return
    report = path.read_text(encoding="utf-8")
    report, count = re.subn(
        r"(?ms)^## Результаты[ \t]*\n.*?(?=^## Выводы[ \t]*$)",
        lambda _: "## Результаты\n\n" + result_section(run, tracks) + "\n\n",
        report,
    )
    if count != 1:
        print("Для обновления отчёта нужны разделы «Результаты» и «Выводы».")
        return
    temporary = path.with_suffix(".md.tmp")
    temporary.write_text(report, encoding="utf-8", newline="\n")
    temporary.replace(path)


tracks = synthesize(run, site_directory)
site_path = build_site(run, site_directory, tracks)
display(FileLink(site_path.relative_to(ROOT).as_posix(), result_html_prefix="Открыть страницу с озвучкой: "))
update_report(run, tracks)


<torch_package_0>.tts_package/multi_acc_v3_package.py:335: SyntaxWarning: invalid escape sequence '\^'
  text = re.sub(r'[^{}]'.format(self.symbols[3:] + '\^'), '', text)


Действие 1: 61.5 с.
Действие 2: 45.3 с.
Действие 3: 52.1 с.


c:\Users\user\Desktop\Agent_Theatre\result\final\site\index.html